In [1]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from wordcloud import WordCloud
from collections import Counter
import time
import logging
from shapely.geometry import Point
from tqdm.auto import tqdm
from scipy import stats
import osmnx as ox
import geopandas as gpd
import json
from collections import defaultdict
import re
import networkx as nx
import warnings



## Данные с большим количеством фичей `17.02.2026`, `24.03.2026`

In [310]:
ATTRIBUTE_REGEX = re.compile(r"\d+")

Items = list[dict]


# Вспомогательная функция для безопасного извлечения вложенных ключей
def get_key_chain(obj, *keys, default=None):
    for key in keys:
        if isinstance(obj, dict):
            obj = obj.get(key)
        elif isinstance(obj, list) and isinstance(key, int):
            try:
                obj = obj[key]
            except IndexError:
                return default
        else:
            return default
            
        if obj is None:
            return default
    return obj


def main(path: str) -> Items:
    global sett
    unique_items = load_json_file(path)
    items = normalize_items(unique_items)
    print(f"normalized items = {len(items)}")
    return items


def load_json_file(file: str) -> Items:
    with open(file, "rt", encoding="utf-8") as file_obj:
        content = file_obj.read().replace("﻿", "")
        try:
            data = json.loads(content)
            if isinstance(data, dict) and "result" in data:
                return data["result"].get("items", [])
            return data
        except Exception:
            print(content[36997921 - 50 : 36997921 + 50])
            raise


def normalize_items(raw_items: Items) -> Items:
    global sett
    items = []

    for item in tqdm(raw_items):
        address = item.get("address_name")
        address_comment = item.get("address_comment")

        if item.get("adm_div") is None:
            continue

        # Извлекаем дополнительные атрибуты
        stat = item.get("stat", {})
        is_advertised = stat.get("is_advertised", None)
        
        flags = item.get("flags", {})
        
        dates = item.get("dates", {})
        created_at = None
        updated_at = None
        
        if 'created_at' in dates:
            created_at = dates['created_at'].split('T')[0]
            
        if 'updated_at' in dates:
            updated_at = dates['updated_at'].split('T')[0]
        org = item.get("org", "")
        if org:
            branch_count = org.get("branch_count", "")
        else:
            branch_count = ""
        # --- ИСПРАВЛЕННЫЙ БЛОК ОБРАБОТКИ РАСПИСАНИЯ ---
        schedule = item.get("schedule", {})
        total_days = 0
        lst_hours = []
        
        for day, info in schedule.items():
            # Проверяем, что значение — это словарь, и в нем есть расписание (отсеиваем 'comment')
            if isinstance(info, dict) and info.get('working_hours'):
                time_range = info['working_hours'][0]
                if 'to' in time_range and 'from' in time_range:
                    total_days += 1
                    h_to = time_range['to'].split(':')
                    h_from = time_range['from'].split(':')
                    
                    end_hour = int(h_to[0])
                    start_hour = int(h_from[0])
                    
                    # Защита от круглосуточных или ночных смен (например, 10:00 - 02:00)
                    if end_hour < start_hour:
                        end_hour += 24
                        
                    h = end_hour - start_hour + (int(h_to[1]) - int(h_from[1])) / 60
                    lst_hours.append(h)
                    
        if total_days > 0:
            hours = sum(lst_hours) / total_days
        else:
            hours = None
        # ----------------------------------------------

        structure_info = item.get("structure_info", {})

        items.append(
            {
                "id": item["id"],
                "building_id": get_key_chain(item, "address", "building_id"),
                "address_comment": address_comment,
                "address": address,
                "lat": get_key_chain(item, "point", "lat"),
                "lon": get_key_chain(item, "point", "lon"),
                "content": extract_content(item),
                "building_floors": get_key_chain(item, "floors", "ground_count"),
                "material": get_key_chain(item, "structure_info", "material"),
                "food_services": ", ".join(
                    attribute["name"]
                    for group in item.get("attribute_groups", [])
                    for attribute in group.get("attributes", [])
                    if attribute.get("tag", "").startswith("food_service")
                ),
                "accessible_entrance": ", ".join(
                    attribute["name"]
                    for group in item.get("attribute_groups", [])
                    for attribute in group.get("attributes", [])
                    if attribute.get("tag", "").startswith("accessible_entrance")
                ),
                "online_training": ", ".join(
                    attribute["name"]
                    for group in item.get("attribute_groups", [])
                    for attribute in group.get("attributes", [])
                    if attribute.get("tag", "").startswith("covid_online_training")
                ),
                "is_advertised": is_advertised,
                "flags": str(flags) if flags else None,
                "temporary_closed": get_key_chain(item, "flags", "temporary_closed"),
                "photos": get_key_chain(item, "flags", "photos"),
                "created_at": created_at,
                "updated_at": updated_at,
                "hours_per_day": hours,
                "branch_count": branch_count,

                "structure_info": str(structure_info) if structure_info else None,
                **{div["type"]: div["name"] for div in item["adm_div"]},
                **extract_titles(item),
                **extract_contacts(item),
                **extract_ratings(item),
                **extract_subs(item),
                **extract_parking(item),
                **extract_metro(item),
            }
        )
        if get_key_chain(item, "flags"):
            for i in get_key_chain(item, "flags").keys():
                sett.add(i)

    return items


def extract_titles(item: dict) -> dict[str, str]:
    if name_ex := item.get("name_ex"):
        return {"title": name_ex.get("short_name") or name_ex["primary"], "extension": name_ex.get("extension")}
    else:
        return {"title": item.get("building_name")}


def extract_contacts(item: dict) -> dict[str, str]:
    contacts = defaultdict(set)

    for contact_group in item.get("contact_groups", []):
        for contact in contact_group.get("contacts", []):
            if contact["type"] == "website":
                contacts["urls"].add(contact["url"])
            elif contact["type"] == "phone":
                contacts["phones"].add(contact["value"])
            elif contact["type"] == "email":
                contacts["emails"].add(contact["value"])

    contacts = {k: ",".join(v) for k, v in contacts.items()}
    return contacts


def extract_ratings(item: dict) -> dict[str, int]:
    ratings = {}
    if reviews := item.get("reviews"):
        ratings["rating"] = reviews.get("general_rating")
        ratings["rating_cnt"] = reviews.get("general_review_count")
    return ratings


def extract_content(item: dict) -> str | None:
    if not (attribute_groups := item.get("attribute_groups")):
        return

    attributes = set()
    for group in attribute_groups:
        for attribute in group["attributes"]:
            if attribute["tag"].startswith("fitness"):
                attributes.add(attribute["name"])

    return ",".join(sorted(attributes))


def extract_subs(item: dict) -> dict[str, str | None]:
    if not (attribute_groups := item.get("attribute_groups")):
        return {}

    years, months = [], []
    price, capacity = None, None
    for group in attribute_groups:
        for attribute in group["attributes"]:
            if attribute.get("tag") == "fitness_year_unltd_subscription_price":
                target = years
            elif attribute.get("tag") == "fitness_month_unltd_subscription_price":
                target = months
            elif attribute.get("tag") == "food_service_capacity":
                nums = ATTRIBUTE_REGEX.findall(attribute["name"])
                if nums: capacity = nums[0]
                continue
            elif attribute.get("tag") == "food_service_avg_price":
                nums = ATTRIBUTE_REGEX.findall(attribute["name"])
                if nums: price = nums[0]
                continue
            else:
                continue

            nums = ATTRIBUTE_REGEX.findall(attribute["name"])
            if nums: target.append(nums[0])

    result = {"avg_price": price, "capacity": capacity}
    if not years and not months:
        return result

    return {"subs_year": ",".join(years), "subs_month": ",".join(months), **result}


def extract_parking(item: dict) -> dict[str, int]:
    # Значения по умолчанию - 0
    result = {
        "parking": 0,
        "free_parking": 0,
        "payed_parking": 0
    }
    
    links = item.get("links", {})
    parkings = links.get("parking", [])
    
    if not parkings:
        return result

    for p in parkings:
        if "capacity" in p:
            try:
                capacity = int(p["capacity"])
                result["parking"] += capacity
                
                # Распределяем по типу (платная/бесплатная)
                is_paid = p.get("is_paid")
                if is_paid is True:
                    result["payed_parking"] += capacity
                elif is_paid is False:
                    result["free_parking"] += capacity
                # Если is_paid не указано (None), то мы не добавляем в платные или бесплатные,
                # но в общей сумме "parking" они уже посчитаны
            except (ValueError, TypeError):
                pass

    return result


def extract_metro(item: dict) -> dict[str, str | int | dict | None]:
    result = {
        "nearest_station": None, 
        "nearest_station_type": None, 
        "nearest_station_d": None, 
        "count_stations": 0,
        "all_stations": {}
    }
    
    links = item.get("links", {})
    stations = links.get("nearest_stations", [])
    
    if not stations:
        return result

    # Отфильтруем те станции, которые являются именно метро (route_types содержит 'metro', 'mcc', 'mcd')
    valid_types = {"metro", "mcc", "mcd"}
    metro_stations = [
        s for s in stations 
        if any(route_type in valid_types for route_type in s.get("route_types", []))
    ]
    
    # Если вдруг по тегам не нашлось, но массив nearest_stations есть, берем его весь
    if not metro_stations:
        metro_stations = stations

    # 1. Заполняем count_metro
    result["count_stations"] = len(metro_stations)
    
    # 2. Заполняем all_metro (словарь {Название: {distance, comment, route_types}})
    all_metro_dict = {}
    for s in metro_stations:
        name = s.get("name")
        dist = s.get("distance")
        if name and dist is not None:
            try:
                dist_int = int(dist)
                # Если станции с таким названием еще нет или найдена более близкая
                if name not in all_metro_dict or dist_int < all_metro_dict[name].get("distance", float('inf')):
                    all_metro_dict[name] = {
                        "distance": dist_int,
                        "comment": s.get("comment"),
                        "route_types": s.get("route_types", [])
                    }
            except (ValueError, TypeError):
                pass
                
    result["all_stations"] = all_metro_dict

    # 3. Находим ближайшую станцию (nearest_metro и nearest_metro_d)
    valid_distances = [s for s in metro_stations if s.get("distance") is not None]
    if valid_distances:
        valid_distances.sort(key=lambda x: int(x.get("distance")))
        nearest = valid_distances[0]
        result["nearest_station"] = nearest.get("name")
        result["nearest_station_type"] = nearest.get("route_types")
        result["nearest_station_d"] = nearest.get("distance")
    
    return result

# # Запуск скрипта
sett = set()
file_path1 = 'fitness_24-03-2026.json' # Укажите здесь ваш файл
file_path2 = 'moscow_fitness_17-02-2026.json' # Укажите здесь ваш файл
data1 = pd.DataFrame(main(file_path1))
data1 = data1.drop_duplicates(subset=['id']).reset_index(drop=True)

data2 = pd.DataFrame(main(file_path2))
data2 = data2.drop_duplicates(subset=['id']).reset_index(drop=True)

  0%|          | 0/2528 [00:00<?, ?it/s]

normalized items = 2528


  0%|          | 0/4193 [00:00<?, ?it/s]

normalized items = 4193


In [311]:
data2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4193 entries, 0 to 4192
Data columns (total 47 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    4193 non-null   object 
 1   building_id           4170 non-null   object 
 2   address_comment       2874 non-null   object 
 3   address               4129 non-null   object 
 4   lat                   4179 non-null   float64
 5   lon                   4179 non-null   float64
 6   content               3823 non-null   object 
 7   building_floors       292 non-null    float64
 8   material              275 non-null    object 
 9   food_services         4193 non-null   object 
 10  accessible_entrance   4193 non-null   object 
 11  online_training       4193 non-null   object 
 12  is_advertised         4184 non-null   object 
 13  flags                 3697 non-null   object 
 14  temporary_closed      75 non-null     object 
 15  photos               

In [312]:
data2['branch_count'] = data2['branch_count'].replace("None", np.nan)
cols = ['branch_count']
data2[cols] = data2[cols].apply(pd.to_numeric, errors='coerce').astype('Int64')

data1['branch_count'] = data1['branch_count'].replace("None", np.nan)
cols = ['branch_count']
data1[cols] = data1[cols].apply(pd.to_numeric, errors='coerce').astype('Int64')
data1['branch_count'].describe()

count       2526.0
mean      8.342439
std      17.779294
min            1.0
25%            1.0
50%            1.0
75%            5.0
max           98.0
Name: branch_count, dtype: Float64

### Обрабатываем информацию об этажах

In [313]:
def to_process_floor(floor):
    if pd.isna(floor) or floor is None:
        return np.nan, np.nan, np.nan
    
    floor = str(floor).strip().lower()
    if not floor:
        return np.nan, np.nan, np.nan
    
    # Заменяем текстовые описания
    floor = floor.replace('цокольный', '-0.5')
    floor = floor.replace('подвал', '-1')
    
    # Разбиваем по перечислениям (запятая или точка с запятой)
    parts = re.split(r'[;,]', floor)
    
    all_floors = []
    
    for part in parts:
        part = part.strip()
        if not part:
            continue
            
        # Проверяем, является ли часть диапазоном: число-число
        # Примеры: "1-3", "-2-0", "1 - 3", "-3--1"
        range_match = re.match(r'^(-?\d+(?:\.\d+)?)\s*-\s*(-?\d+(?:\.\d+)?)$', part)
        
        if range_match:
            start = float(range_match.group(1))
            end = float(range_match.group(2))
            
            # Если оба целые — раскрываем диапазон полностью
            if start == int(start) and end == int(end):
                start, end = int(start), int(end)
                step = 1 if start <= end else -1
                all_floors.extend(range(start, end + step, step))
            else:
                # Для дробных (например, -0.5-2) просто добавляем границы
                all_floors.extend([start, end])
        else:
            # Одиночное число (включая отрицательные)
            try:
                num = float(part)
                all_floors.append(num)
            except ValueError:
                continue
    
    if not all_floors:
        return np.nan, np.nan, np.nan
    
    f_min = min(all_floors)
    f_max = max(all_floors)
    f_count = len(set(all_floors))  # Уникальные этажи
    
    return float(f_min), float(f_max), int(f_count)

# Применение
data1['object_floor'] = data1['address_comment'].apply(
    lambda x: x.split('этаж')[0].split(';')[-1] if x and 'этаж' in x else None
)
data1[['floor_min', 'floor_max', 'floor_count']] = data1['object_floor'].apply(
    lambda x: pd.Series(to_process_floor(x))
)

# Применение
data2['object_floor'] = data2['address_comment'].apply(
    lambda x: x.split('этаж')[0].split(';')[-1] if x and 'этаж' in x else None
)
data2[['floor_min', 'floor_max', 'floor_count']] = data2['object_floor'].apply(
    lambda x: pd.Series(to_process_floor(x))
)

In [314]:
data1[['object_floor','floor_min', 'floor_max', 'floor_count']].value_counts().head()

object_floor  floor_min  floor_max  floor_count
1              1.0        1.0       1.0            684
2              2.0        2.0       1.0            286
цокольный     -0.5       -0.5       1.0            199
3              3.0        3.0       1.0            181
4              4.0        4.0       1.0             94
Name: count, dtype: int64

In [315]:
data2[['object_floor','floor_min', 'floor_max', 'floor_count']].value_counts().head()

object_floor  floor_min  floor_max  floor_count
1              1.0        1.0       1.0            889
2              2.0        2.0       1.0            565
цокольный     -0.5       -0.5       1.0            351
3              3.0        3.0       1.0            286
4              4.0        4.0       1.0            116
Name: count, dtype: int64

In [316]:
data1.to_excel('moscow_fitness_24-03-2026.xlsx', index=False)
data2.to_excel('moscow_fitness_17-02-2026.xlsx', index=False)

## Обрабатываем выгрузки фитнесов из 2ГИС

In [317]:
folder_path = '.'  # укажите ваш путь
dict_fitness = {}
sett = set()
lst_fitness = []  # список для хранения датафреймов

for name in os.listdir(folder_path):
    if 'fitness' in name and name.endswith(('.xlsx', '.xls')):
        # Извлекаем дату из имени файла
        y = name[name.rfind('_')+1:name.rfind('.')]
        dict_fitness[y] = name
        print(y)
        
        # Читаем Excel файл
        df = pd.read_excel(name)
        content = list(df.columns)
        print(content)
        print()
        
        # Добавляем столбец 'date'
        df['date'] = y
        
        # Собираем уникальные колонки
        [sett.add(i) for i in content]
        
        # Добавляем датафрейм в список
        lst_fitness.append(df)

# Проверяем все найденные колонки
print(f"Все уникальные колонки {len(sett)} шт.: {sorted(sett)}")

# Конкатенируем все датафреймы
if lst_fitness:
    df_concat = pd.concat(lst_fitness, ignore_index=True)
    print(f"\nИтоговый датафрейм: {df_concat.shape}")
else:
    print("Файлы не найдены")

01-08-2023
['id', 'building_id', 'building_type', 'title', 'extension', 'region', 'city', 'address', 'phones', 'emails', 'urls', 'rating', 'rating_cnt', 'content']

07-02-2025
['id', 'building_id', 'building_type', 'lat', 'lon', 'region', 'city', 'district', 'living_area', 'address', 'title', 'extension', 'phones', 'emails', 'urls', 'rating', 'rating_cnt', 'content', 'subs_year', 'subs_month', 'avg_price', 'capacity', 'floors', 'material']

13-12-2024
['id', 'building_id', 'building_type', 'lat', 'lon', 'region', 'city', 'district', 'living_area', 'address', 'title', 'extension', 'phones', 'emails', 'urls', 'rating', 'rating_cnt', 'content', 'subs_year', 'subs_month', 'avg_price', 'capacity', 'floors', 'material']

17-02-2026
['id', 'building_id', 'address_comment', 'address', 'lat', 'lon', 'content', 'building_floors', 'material', 'food_services', 'accessible_entrance', 'online_training', 'is_advertised', 'flags', 'temporary_closed', 'photos', 'created_at', 'updated_at', 'hours_per_da

In [318]:
df_concat.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33086 entries, 0 to 33085
Data columns (total 54 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    33077 non-null  object 
 1   building_id           33003 non-null  float64
 2   building_type         16914 non-null  object 
 3   title                 33060 non-null  object 
 4   extension             27980 non-null  object 
 5   region                33086 non-null  object 
 6   city                  29948 non-null  object 
 7   address               32812 non-null  object 
 8   phones                31783 non-null  object 
 9   emails                2833 non-null   object 
 10  urls                  25866 non-null  object 
 11  rating                20175 non-null  object 
 12  rating_cnt            20175 non-null  float64
 13  content               25804 non-null  object 
 14  date                  33086 non-null  object 
 15  lat                

In [319]:
# df_concat = df_concat[['id', 'building_id', 'building_type', 'title', 'extension', 'region',
#        'city', 'address', 'phones', 'emails', 'urls', 'rating', 'rating_cnt',
#        'content', 'date', 'lat', 'lon', 'district', 'living_area', 'subs_year',
#        'subs_month', 'floors', 'material']]
# df_concat.info()

In [320]:
for d in df_concat['date'].unique():
    print(d)
    dump = len(df_concat[df_concat['date']==d])
    clear = len(df_concat[df_concat['date']==d].drop_duplicates(subset=['building_id', 'title', 'lat', 'lon', 'address']))
    print(f'{dump-clear} dups')
    print()

01-08-2023
12 dups

07-02-2025
18 dups

13-12-2024
7 dups

17-02-2026
311 dups

17-11-2025
18 dups

22-02-2024
5 dups

24-03-2026
548 dups

25-07-2024
17 dups

27-08-2025
2926 dups



In [321]:
print(df_concat.shape)
df_concat = df_concat.drop_duplicates(subset=['building_id', 'title', 'lat', 'lon', 'address', 'date']).reset_index(drop=True)
print(df_concat.shape)

(33086, 54)
(29224, 54)


### Обрабатываем рейтинг, который был превращен в excel в дату

In [322]:
df_concat['rating'].value_counts()

rating
5                      8051
4                       563
3                       495
1                       378
4.9                     368
                       ... 
1.8                       3
2024-09-01 00:00:00       3
1.6                       2
1.1                       1
2024-04-01 00:00:00       1
Name: count, Length: 114, dtype: int64

In [323]:
def to_process_rating(value):
    # Проверяем на None/nan, но НЕ отбрасываем 0 и 0.0!
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    
    if str(value).lower() == 'nan':
        return None
        
    value = str(value).strip()
    
    # Если это число с точкой или целое число 0-5
    if value.replace('.', '', 1).replace('-', '', 1).isdigit() and value.count('.') <= 1:
        try:
            f = float(value)
            if 0 <= f <= 5:
                return f
        except:
            pass
    
    # Если это дата (содержит '-')
    elif '-' in value:
        date_part = value.split()[0]
        parts = date_part.split('-')
        if len(parts) >= 3:
            try:
                year, month, day = parts[0], parts[1], parts[2]
                return float(f"{int(day)}.{int(month)}")
            except:
                return None
    
    return None

df_concat['rating'] = df_concat['rating'].apply(to_process_rating)

In [324]:
df_concat['rating'].describe()

count    17688.000000
mean         4.395732
std          0.912680
min          0.000000
25%          4.100000
50%          4.900000
75%          5.000000
max          5.000000
Name: rating, dtype: float64

In [325]:
df_concat.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29224 entries, 0 to 29223
Data columns (total 54 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    29215 non-null  object 
 1   building_id           29152 non-null  float64
 2   building_type         16855 non-null  object 
 3   title                 29198 non-null  object 
 4   extension             24677 non-null  object 
 5   region                29224 non-null  object 
 6   city                  26439 non-null  object 
 7   address               28994 non-null  object 
 8   phones                28046 non-null  object 
 9   emails                2541 non-null   object 
 10  urls                  22745 non-null  object 
 11  rating                17688 non-null  float64
 12  rating_cnt            17688 non-null  float64
 13  content               22422 non-null  object 
 14  date                  29224 non-null  object 
 15  lat                

In [326]:
df_concat['date'].value_counts()

date
25-07-2024    5590
17-02-2026    3882
27-08-2025    3449
07-02-2025    3141
17-11-2025    3028
13-12-2024    3026
22-02-2024    2702
01-08-2023    2426
24-03-2026    1980
Name: count, dtype: int64

### Обрезаем по границе Москвы

In [327]:
len(df_concat)-len(df_concat[['lat', 'lon']].dropna())

2472

In [328]:
df_concat[df_concat['lat'].isna()]['date'].value_counts()

date
01-08-2023    2426
17-02-2026       8
07-02-2025       6
25-07-2024       6
24-03-2026       6
17-11-2025       5
13-12-2024       5
22-02-2024       5
27-08-2025       5
Name: count, dtype: int64

In [330]:
gdf_points = gpd.GeoDataFrame(
    df_concat,
    geometry=gpd.points_from_xy(df_concat['lon'], df_concat['lat']),
    crs="EPSG:4326"
)

moscow_boundary = gpd.read_file('moscow_boundary.gpkg')

if moscow_boundary.crs != gdf_points.crs:
    moscow_boundary = moscow_boundary.to_crs(gdf_points.crs)

gdf_clipped = gpd.clip(gdf_points, moscow_boundary)

gdf_clipped['date'].value_counts()

date
25-07-2024    3484
17-11-2025    1934
27-08-2025    1887
17-02-2026    1883
07-02-2025    1857
13-12-2024    1817
24-03-2026    1802
22-02-2024    1725
Name: count, dtype: int64

In [331]:
gdf_clipped.columns

Index(['id', 'building_id', 'building_type', 'title', 'extension', 'region',
       'city', 'address', 'phones', 'emails', 'urls', 'rating', 'rating_cnt',
       'content', 'date', 'lat', 'lon', 'district', 'living_area', 'subs_year',
       'subs_month', 'avg_price', 'capacity', 'floors', 'material',
       'address_comment', 'building_floors', 'food_services',
       'accessible_entrance', 'online_training', 'is_advertised', 'flags',
       'temporary_closed', 'photos', 'created_at', 'updated_at',
       'hours_per_day', 'branch_count', 'structure_info', 'country', 'parking',
       'free_parking', 'payed_parking', 'nearest_station',
       'nearest_station_type', 'nearest_station_d', 'count_stations',
       'all_stations', 'district_area', 'settlement', 'object_floor',
       'floor_min', 'floor_max', 'floor_count', 'geometry'],
      dtype='object')

In [332]:
gdf_clipped[list(gdf_clipped.columns[:-1])].to_excel('merged_data.xlsx', index=False)
gdf_clipped.to_file('merged_data.gpkg', index=False)

### Работаем с `extension`
Ручная проверка точек по отношению к фитнесу. Если категория соответствует фитнесу, то не рассматриваем точку, а если не соответствует, то просматриваем детально объекты, роверяем фитнес это или нет. Если нет, то удаляем такие объекты.

In [338]:
#pd.DataFrame(df_concat['extension'].value_counts()).reset_index().to_excel('fitness_types.xlsx', index = False)

In [339]:
to_del_lst = [
    'частная школа-детский сад',
    'детский центр',
    'центр выздоровления',
    'культурный центр',
    'студия красоты',
    'магазин спортивного оборудования',
    'частная школа',
    'частный детский сад',
    'центр биохакинга-миофлекс',
    'отделения Спутник и Рекорд',
    'офис',
    'центр детского развития',
    'центр косметологии и аппаратной коррекции фигуры для мужчин и женщин',
    'территориальное управление Ивановское',
    'театральная студия',
    'центр раннего развития',
    'студия аппаратной косметологии',
    'частная школа-детский сад']

length_before=len(df_concat)
df_concat = df_concat[~df_concat['extension'].isin(to_del_lst)].reset_index(drop=True)
length_after=len(df_concat)
print(f'{length_before} -> {length_after}')

16389 -> 16349


#### Удаляем данные с пропусками в ключевых атрибутах

In [340]:
length_before=len(df_concat)
df_concat=df_concat.dropna(subset='title').reset_index(drop=True)
length_after=len(df_concat)
print(f'{length_before} -> {length_after}')

16349 -> 16340


### Че-то агрегируем

In [341]:
df_concat[['id', 'building_id']].value_counts()

id                                                                                                                                                         building_id 
4504127908385537                                                                                                                                           4.504235e+15    4
4504127908455138.0                                                                                                                                         4.504235e+15    4
70000001066386360.0                                                                                                                                        4.504235e+15    4
70000001046148240.0                                                                                                                                        4.504235e+15    4
4504127908455347                                                                                                                            

In [342]:
df_concat['building_id'].value_counts()

building_id
4.504235e+15    50
4.504235e+15    48
7.003008e+16    41
4.504235e+15    37
4.504235e+15    36
                ..
4.504235e+15     1
4.504235e+15     1
4.504235e+15     1
4.504235e+15     1
4.504235e+15     1
Name: count, Length: 3106, dtype: int64

In [343]:
df_concat['id'].value_counts()

id
70000001060615592.0                                                                                                                                            6
70000001089603328                                                                                                                                              4
4504128908519697.0                                                                                                                                             4
70000001068932400                                                                                                                                              4
4504127908543170                                                                                                                                               4
                                                                                                                                                              ..
70000001069123104_vlrl43e6dBdB9

## Чистим данные в ключевых столбцах для агрегации

In [344]:
def to_clean_title(text):
    if text:
        text=str(text).lower()
        for i in '.- &,_!?*()`':
            text=text.replace(i,'')
        return text
    else:
        return None

def to_clean_address(text):
    if text:
        text=str(text).lower()
        for i in '.- &,_!?*()`':
            text=text.replace(i,'')
        return text
    else:
        return None
        
def to_clean_district(text):
    if text:
        text=str(text).lower()
        for i in '.- &,_!?*()`':
            text=text.replace(i,'')
        text=text.replace('ё', 'е')
        text=text.replace('троицкгорокруг', 'троицк')
        return text
    else:
        return None  
        
df_concat['clean_title'] = df_concat['title'].apply(to_clean_title)
df_concat['clean_address'] = df_concat['address'].apply(to_clean_address)
df_concat['clean_district']=df_concat['district'].apply(to_clean_district)
df_concat['clean_district']=df_concat.apply(lambda row: 'солнцево' if row['address']=='Родниковая улица, 9а к4' else row['clean_district'], axis=1)
df_concat['district']=df_concat.apply(lambda row: 'Солнцево' if row['address']=='Родниковая улица, 9а к4' else row['district'], axis=1)

print(f"various titles {len(df_concat['title'].value_counts())} -> {len(df_concat['clean_title'].value_counts())}")
print(f"various addresses {len(df_concat['address'].value_counts())} -> {len(df_concat['clean_address'].value_counts())}") # из-за None
print(f"various districts {len(df_concat['district'].value_counts())} -> {len(df_concat['clean_district'].value_counts())}")

df_concat['lat_round'] = df_concat['lat'].round(3).astype(str)
df_concat['lon_round'] = df_concat['lon'].round(3).astype(str)

various titles 2988 -> 2844
various addresses 3210 -> 3211
various districts 144 -> 132


In [346]:
bui_id=int(pd.DataFrame(df_concat['building_id'].value_counts()).reset_index()['building_id'][1])
df_concat[df_concat['building_id']==bui_id]

,id,building_id,building_type,title,extension,region,city,address,phones,emails,...,settlement,object_floor,floor_min,floor_max,floor_count,clean_title,clean_address,clean_district,lat_round,lon_round
2594,70000001112414754_cwm2kdzBdBdB9A827J7H2J2JHGII...,4.504235e+15,NaN,Cherry fitness,NaN,Москва,Москва,"Профсоюзная улица, 56",+79933345342,NaN,...,NaN,NaN,NaN,NaN,NaN,cherryfitness,профсоюзнаяулица56,обручевский,55.67,37.553
2595,70000001093847990,4.504235e+15,Многофункциональный комплекс,Fitjumping,студия джампинг-фитнеса,Москва,Москва,"Профсоюзная улица, 56",+74950850054,NaN,...,NaN,NaN,NaN,NaN,NaN,fitjumping,профсоюзнаяулица56,обручевский,55.67,37.553
2596,70000001093847990_6j9egt37dBdB9A826J7H1BBJHGI2...,4.504235e+15,NaN,Fitjumping,студия фитнеса,Москва,Москва,"Профсоюзная улица, 56",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,fitjumping,профсоюзнаяулица56,обручевский,55.67,37.553
2597,70000001093847992.0,4.504235e+15,NaN,Fitjumping,студия джампинг-фитнеса,Москва,Москва,"Профсоюзная улица, 56",+74993489945,NaN,...,NaN,NaN,NaN,NaN,NaN,fitjumping,профсоюзнаяулица56,обручевский,55.67,37.553
2598,70000001093847992.0,4.504235e+15,Многофункциональный комплекс,Fitjumping,студия джампинг-фитнеса,Москва,Москва,"Профсоюзная улица, 56",+74950850054,NaN,...,NaN,NaN,NaN,NaN,NaN,fitjumping,профсоюзнаяулица56,обручевский,55.67,37.553
2609,70000001035577618_f33q22785BG2J1HA01I2GGG0in68...,4.504235e+15,Многофункциональный комплекс,Topstretching,студия растяжки и функционального тренинга,Москва,Москва,"Профсоюзная улица, 56","+74951727990,,,4",NaN,...,NaN,NaN,NaN,NaN,NaN,topstretching,профсоюзнаяулица56,обручевский,55.67,37.553
2610,70000001056468811_6hudy7785BG23H1J7188G2GGwlko...,4.504235e+15,Многофункциональный комплекс,ProstoShpagat,студия растяжки,Москва,Москва,"Профсоюзная улица, 56",+79680051005,NaN,...,NaN,NaN,NaN,NaN,NaN,prostoshpagat,профсоюзнаяулица56,обручевский,55.67,37.553
2611,70000001065986002_9Byrch785BG2CH1J7188G2GGrpx6...,4.504235e+15,Многофункциональный комплекс,Corpssain,студия растяжки,Москва,Москва,"Профсоюзная улица, 56",+79857952707,NaN,...,NaN,NaN,NaN,NaN,NaN,corpssain,профсоюзнаяулица56,обручевский,55.67,37.553
2612,70000001065986002,4.504235e+15,Многофункциональный комплекс,Corpssain,студия растяжки,Москва,Москва,"Профсоюзная улица, 56",+79857952707,NaN,...,NaN,NaN,NaN,NaN,NaN,corpssain,профсоюзнаяулица56,обручевский,55.67,37.553
2613,70000001056468811_hj2j67e5dBdB9A829J7H9CI3HG2I...,4.504235e+15,NaN,ProstoShpagat,студия спорта и растяжки,Москва,Москва,"Профсоюзная улица, 56",+79680051005,NaN,...,NaN,11,11.0,11.0,1.0,prostoshpagat,профсоюзнаяулица56,обручевский,55.67,37.553


In [373]:
df_concat[df_concat['address']=='Родниковая улица, 9а к4'][['date','lat','lon','district']]

,date,lat,lon,district
504,22-02-2024,55.633849,37.417107,Солнцево
505,07-02-2025,55.633849,37.417107,Солнцево
506,17-11-2025,55.633849,37.417107,Солнцево
527,13-12-2024,55.633849,37.417107,Солнцево
528,24-03-2026,55.633849,37.417107,Солнцево
529,17-02-2026,55.633849,37.417107,Солнцево
530,25-07-2024,55.633849,37.417107,Солнцево
531,27-08-2025,55.633849,37.417107,Солнцево


In [374]:
df_concat[['clean_title', 'lat', 'lon']].value_counts()

clean_title               lat        lon      
рассвет1905               55.762455  37.568896    14
beflex                    55.799303  37.718131    10
олимпийскаядеревня80      55.676280  37.473190     9
#slimfitclub              55.725724  37.545447     8
юналайф                   55.609883  37.604422     8
                                                  ..
академиячемпионов         55.728113  37.416428     1
академияфитнесаипилатеса  55.747794  37.586897     1
                          55.747638  37.586602     1
янаспорт                  55.834031  37.659157     1
ямамапапа                 55.853870  37.346493     1
Name: count, Length: 5734, dtype: int64

In [351]:
df_concat[['clean_title', 'lat_round', 'lon_round']].value_counts()

clean_title    lat_round  lon_round
xfit           55.783     37.56        24
xline          55.805     37.781       16
центральный    55.744     37.666       16
рассвет1905    55.762     37.569       14
палестраспорт  55.794     37.516       14
                                       ..
юговосток      55.703     37.822        1
               55.709     37.765        1
южный          55.632     37.762        1
               55.678     37.72         1
+38°           55.566     37.57         1
Name: count, Length: 5108, dtype: int64

In [352]:
df_concat[['clean_title', 'clean_address']].value_counts()

clean_title   clean_address              
xfit          ленинградскийпроспект31аст1    24
ddxfitness    кронштадтскийбульвар3а         16
kibrologygym  береговойпроезд5ак1            15
kulturafit    нижегородскаяулица32ст3        10
proartfit     мосфильмовскаяулица25          10
                                             ..
+38°          улицагрина7б                    1
ямакаси       мосфильмовскаяулица53           1
              каширскоешоссе39                1
ягюсинганрю   калашныйпереулок10ст1           1
ягуардм       4якабельнаяулица2ст1а           1
Name: count, Length: 5022, dtype: int64

In [353]:
df_concat[df_concat['clean_address']=='nan'][['title', 'lat', 'lon', 'district','date']]

,title,lat,lon,district,date
53,Stop for fit,55.500110,37.517494,Щербинка,17-11-2025
54,Stop for fit,55.500110,37.517494,Щербинка,27-08-2025
55,Stop for fit,55.500110,37.517494,Щербинка,07-02-2025
56,Stop for fit,55.500110,37.517494,Щербинка,17-02-2026
148,Боевое самбо,55.528646,37.478160,Южное Бутово,25-07-2024
149,Гошин Каратэ Джитсу,55.528646,37.478160,Южное Бутово,25-07-2024
269,Stop for fit,55.541220,37.457030,Коммунарка,07-02-2025
1223,Тренажёрный зал,55.629252,37.379935,Ново-Переделкино,24-03-2026
1224,Тренажёрный зал,55.629252,37.379935,Ново-Переделкино,17-02-2026
1922,Битца,55.643250,37.584458,Чертаново Северное,13-12-2024


In [356]:
df_concat[['clean_title','clean_address', 'district']].value_counts()

clean_title   clean_address                district          
xfit          ленинградскийпроспект31аст1  Беговой               24
ddxfitness    кронштадтскийбульвар3а       Головинский           16
kibrologygym  береговойпроезд5ак1          Филевский парк        13
crossfit1905  улицалужники24               Хамовники             10
proartfit     мосфильмовскаяулица25        Раменки               10
                                                                 ..
явь           посёлокзнамяоктября29        Щербинка               1
ягуар         дегунинскаяулица17           Западное Дегунино      1
              дубнинскаяулица6             Восточное Дегунино     1
ягуардм       4якабельнаяулица2ст1а        Лефортово              1
юность        зеленоградк813               Старое Крюково         1
Name: count, Length: 5195, dtype: int64

In [358]:
gdf_clipped['date'].value_counts()

date
25-07-2024    3484
17-11-2025    1934
27-08-2025    1887
17-02-2026    1883
07-02-2025    1857
13-12-2024    1817
24-03-2026    1802
22-02-2024    1725
Name: count, dtype: int64

In [359]:
df_concat.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16340 entries, 0 to 16339
Data columns (total 59 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    16340 non-null  object 
 1   building_id           16340 non-null  float64
 2   building_type         8848 non-null   object 
 3   title                 16340 non-null  object 
 4   extension             13881 non-null  object 
 5   region                16340 non-null  object 
 6   city                  15963 non-null  object 
 7   address               16301 non-null  object 
 8   phones                15798 non-null  object 
 9   emails                1815 non-null   object 
 10  urls                  13991 non-null  object 
 11  rating                10822 non-null  float64
 12  rating_cnt            10822 non-null  float64
 13  content               12666 non-null  object 
 14  date                  16340 non-null  object 
 15  lat                

In [102]:
'id', 'building_id', 'building_type', 'title', 'extension', 'region',
       'city', 'address', 'phones', 'emails', 'urls', 'rating', 'rating_cnt',
       'content', 'date', 'lat', 'lon', 'district', 'living_area', 'subs_year',
       'subs_month', 'floors', 'material',
       'address_comment', 'building_floors',
       'accessible_entrance', 'online_training', 'is_advertised', 'flags',
       'temporary_closed', 'photos', 'created_at', 'updated_at',
       'hours_per_day', 'structure_info', 'country', 'parking', 'free_parking',
       'payed_parking', 'nearest_station', 'nearest_station_type',
       'nearest_station_d', 'count_stations', 'all_stations', 'district_area',
       'settlement', 'object_floor', 'floor_min', 'floor_max', 'floor_count'

Index(['id', 'building_id', 'building_type', 'title', 'extension', 'region',
       'city', 'address', 'phones', 'emails', 'urls', 'rating', 'rating_cnt',
       'content', 'date', 'lat', 'lon', 'district', 'living_area', 'subs_year',
       'subs_month', 'avg_price', 'capacity', 'floors', 'material',
       'address_comment', 'building_floors', 'food_services',
       'accessible_entrance', 'online_training', 'is_advertised', 'flags',
       'temporary_closed', 'photos', 'created_at', 'updated_at',
       'hours_per_day', 'structure_info', 'country', 'parking', 'free_parking',
       'payed_parking', 'nearest_station', 'nearest_station_type',
       'nearest_station_d', 'count_stations', 'all_stations', 'district_area',
       'settlement', 'object_floor', 'floor_min', 'floor_max', 'floor_count'],
      dtype='object')

In [377]:
df = df_concat.copy()
# Приводим даты к формату datetime для корректной сортировки и сравнения
df['date'] = pd.to_datetime(df['date'], format='%d-%m-%Y')


# Округляем координаты для подстраховки (4 знака ~ 111 метров точности)
df['lat_round'] = df['lat'].round(3).astype(str)
df['lon_round'] = df['lon'].round(3).astype(str)

# Создаем уникальный синтетический ключ
# Если адреса нет ('nan'), используем округленные координаты
df['club_key'] = df.apply(
    lambda row: f"{row['clean_title']}_{row['clean_address']}" 
    if row['clean_address'] != 'nan' 
    else f"{row['clean_title']}_{row['lat']}_{row['lon']}", 
    axis=1
)
# Все строки, которые имеют дубликаты по club_key + date
duplicates = df[df.duplicated(subset=['club_key', 'date'], keep=False)]

# Сортируем для наглядности, чтобы дубликаты шли подряд
duplicates = duplicates.sort_values(by=['club_key', 'date', 'clean_title'])
print(len(df))
print(len(df.drop_duplicates(subset=['club_key', 'date'])))

16340
16266


In [378]:
duplicates[['club_key', 'district', 'date', 'content']].value_counts()

club_key                                                     district             date        content                                                                                                                                                            
ddxfitness_кронштадтскийбульвар3а                            Головинский          2024-02-22  TRX тренировки,Аэробика,Возможны разовые посещения,Годовой абонемент от 22400 ₽,Зумба,Йога,Месячный абонемент от 1700 ₽,Персональный тренер,Пилатес,Сауны,Стретчинг    2
botanicfitness_проездсеребрякова1/2                          Свиблово             2025-08-27  Боевые искусства,Кроссфит,Персональный тренер,Сауны,Фитнес-бокс                                                                                                        2
академияфитнесаипилатеса_улицаарбат51ст1                     Арбат                2025-08-27  Пилатес,Стретчинг                                                                                                         

In [379]:
duplicates.sort_values(by=['club_key','date','rating','rating_cnt','subs_year','subs_month','content'],
                       ascending=[True, True, False, False, False, False, False])[['club_key','date','rating','rating_cnt','subs_year','subs_month','content']].to_excel('duplicates.xlsx', index=False)

In [381]:
duplicates[['club_key', 'district', 'date']].value_counts()[:30]

club_key                             district        date      
kibrologygym_береговойпроезд5ак1     Филевский парк  2024-07-25    3
kibrology_береговойпроезд5ак1        Филевский парк  2025-08-27    3
kibrologygym_береговойпроезд5ак1     Филевский парк  2024-02-22    3
xfit_ленинградскийпроспект31аст1     Беговой         2026-03-24    3
                                                     2026-02-17    3
                                                     2025-11-17    3
                                                     2024-02-22    3
                                                     2024-12-13    3
                                                     2024-07-25    3
                                                     2025-08-27    3
                                                     2025-02-07    3
kibrologygym_береговойпроезд5ак1     Филевский парк  2025-02-07    3
                                                     2024-12-13    3
ddxfitness_кронштадтскийбульвар3а    Го

In [193]:
duplicates[['club_key', 'district', 'date']].value_counts()[30:]

club_key                                                     district             date      
freshstretching_киевскаяулица2                               Дорогомилово         2024-07-25    2
gtfit_улицалужники24ст21                                     Хамовники            2025-02-07    2
idealfitness_конныйпереулок4                                 Якиманка             2025-08-27    2
freshstretching_киевскаяулица2                               Дорогомилово         2024-02-22    2
ddxfitness_кронштадтскийбульвар3а                            Головинский          2026-02-17    2
                                                                                  2026-03-24    2
fastbody_проспектмира26ст1                                   Мещанский            2024-07-25    2
ddxfitness_кронштадтскийбульвар3а                            Головинский          2025-08-27    2
                                                                                  2025-02-07    2
proartfit_мосфильмовскаяу

In [206]:
df_concat.columns

Index(['id', 'building_id', 'building_type', 'title', 'extension', 'region',
       'city', 'address', 'phones', 'emails', 'urls', 'rating', 'rating_cnt',
       'content', 'date', 'lat', 'lon', 'district', 'living_area', 'subs_year',
       'subs_month', 'avg_price', 'capacity', 'floors', 'material',
       'address_comment', 'building_floors', 'food_services',
       'accessible_entrance', 'online_training', 'is_advertised', 'flags',
       'temporary_closed', 'photos', 'created_at', 'updated_at',
       'hours_per_day', 'structure_info', 'country', 'parking', 'free_parking',
       'payed_parking', 'nearest_station', 'nearest_station_type',
       'nearest_station_d', 'count_stations', 'all_stations', 'district_area',
       'settlement', 'object_floor', 'floor_min', 'floor_max', 'floor_count',
       'clean_title', 'clean_address', 'lat_round', 'lon_round',
       'district_clean', 'clean_district'],
      dtype='object')

In [211]:
df_concat['created_at'].dropna().min()

'2002-06-25'

In [212]:
df_concat['updated_at'].dropna().min()

'2022-07-28'

In [213]:
df_concat['updated_at'].dropna()

5        2026-01-02
21       2026-01-14
27       2026-03-20
28       2025-05-15
34       2025-05-15
            ...    
16326    2026-01-14
16331    2026-01-22
16332    2026-01-22
16335    2026-01-22
16336    2026-01-22
Name: updated_at, Length: 3676, dtype: object

In [219]:
df_concat.columns

Index(['id', 'building_id', 'building_type', 'title', 'extension', 'region',
       'city', 'address', 'phones', 'emails', 'urls', 'rating', 'rating_cnt',
       'content', 'date', 'lat', 'lon', 'district', 'living_area', 'subs_year',
       'subs_month', 'avg_price', 'capacity', 'floors', 'material',
       'address_comment', 'building_floors', 'food_services',
       'accessible_entrance', 'online_training', 'is_advertised', 'flags',
       'temporary_closed', 'photos', 'created_at', 'updated_at',
       'hours_per_day', 'structure_info', 'country', 'parking', 'free_parking',
       'payed_parking', 'nearest_station', 'nearest_station_type',
       'nearest_station_d', 'count_stations', 'all_stations', 'district_area',
       'settlement', 'object_floor', 'floor_min', 'floor_max', 'floor_count',
       'clean_title', 'clean_address', 'lat_round', 'lon_round',
       'district_clean', 'clean_district'],
      dtype='object')

In [226]:
df_concat['living_area'].value_counts()

living_area
Люберецкие Поля м-н    75
Центральный м-н        68
Матвеевское м-н        63
Некрасовка пос.        50
3-й м-н                42
                       ..
Барыши м-н              1
Кожухово пос.           1
Ухтомский м-н           1
ДСК им. Ларина          1
10-й м-н                1
Name: count, Length: 76, dtype: int64

In [362]:
df_concat.sort_values(by='subs_year')[['title','district', 'content', 'date','subs_year','subs_month']][:20]

,title,district,content,date,subs_year,subs_month
4665,Levita,Арбат,"Возможны разовые посещения,Годовой абонемент о...",17-11-2025,450.0,450.0
4667,Levita,Арбат,"Возможны разовые посещения,Годовой абонемент о...",17-02-2026,450.0,450.0
4668,Levita,Арбат,"Возможны разовые посещения,Годовой абонемент о...",27-08-2025,450.0,450.0
7036,Fitjumping,Тимирязевский,"Годовой абонемент от 1490 ₽,Джампинг,Месячный ...",07-02-2025,1490.0,1490.0
6725,Fitjumping,Тимирязевский,"Аэробика,Годовой абонемент от 1490 ₽,Джампинг,...",13-12-2024,1490.0,1490.0
6723,Fitjumping,Тимирязевский,"Аэробика,Годовой абонемент от 1490 ₽,Джампинг,...",25-07-2024,1490.0,1490.0
6722,Fitjumping,Тимирязевский,"Годовой абонемент от 1490 ₽,Джампинг,Месячный ...",27-08-2025,1490.0,1490.0
6146,Powerhouse Gym Сокол,Сокол,"TRX тренировки,Аэробика,Боевые искусства,Возмо...",27-08-2025,1500.0,1500.0
6150,Powerhouse Gym Сокол,Сокол,"TRX тренировки,Аэробика,Боевые искусства,Возмо...",07-02-2025,1500.0,1500.0
6152,Powerhouse Gym Сокол,Сокол,"TRX тренировки,Аэробика,Боевые искусства,Возмо...",13-12-2024,1500.0,1500.0


In [365]:
df_concat.dropna(subset='subs_year')[['date']].value_counts()

date      
27-08-2025    528
17-11-2025    524
07-02-2025    506
25-07-2024    497
17-02-2026    496
13-12-2024    488
24-03-2026    425
Name: count, dtype: int64

In [367]:
# Список нужных столбцов
columns_to_count = ['date','rating', 'rating_cnt', 'content', 'subs_year', 'subs_month']

# Группировка по дате и подсчет непустых значений
result = df_concat.groupby('date')[columns_to_count].count()

result

,date,rating,rating_cnt,content,subs_year,subs_month
date,,,,,,
07-02-2025,1854,1286,1286,1657,506,959
13-12-2024,1815,1272,1272,1517,488,858
17-02-2026,1879,1367,1367,1623,496,956
17-11-2025,1931,1390,1390,1685,524,958
22-02-2024,1722,1188,1188,1402,0,0
24-03-2026,1797,1308,1308,1648,425,1046
25-07-2024,3457,1670,1670,1453,497,829
27-08-2025,1885,1341,1341,1681,528,973


## Агрегируем

In [385]:
# --- ШАГ 1: Предобработка и нормализация ---
df = df_concat.copy()
# Приводим даты к формату datetime для корректной сортировки и сравнения
df['date'] = pd.to_datetime(df['date'], format='%d-%m-%Y')


# # Округляем координаты для подстраховки (4 знака ~ 111 метров точности)
# df['lat_round'] = df['lat'].round(3).astype(str)
# df['lon_round'] = df['lon'].round(3).astype(str)

# Создаем уникальный синтетический ключ
# Если адреса нет ('nan'), используем координаты
df['club_key'] = df.apply(
    lambda row: f"{row['clean_title']}_{row['clean_address']}" 
    if row['clean_address'] != 'nan' 
    else f"{row['clean_title']}_{row['lat']}_{row['lon']}", 
    axis=1
)

# Удаление внутридневных дубликатов ---
# Если в один день один и тот же клуб спарсился дважды, оставляем тот, где больше данных (где есть рейтинг, отзывы, цены абонементов, услуги)
df = df.sort_values(by=['club_key','date','rating','rating_cnt','subs_year','subs_month','content'],
                    ascending=[True, True, False, False, False, False, False])
df = df.drop_duplicates(subset=['club_key', 'date'], keep='first')


# Определяем глобальные границы парсинга
min_global_date = df['date'].min()
max_global_date = df['date'].max()

# Агрегируем временную информацию
timeline = df.groupby('club_key').agg(
    first_seen=('date', 'min'),
    last_seen=('date', 'max'),
    observations_count=('date', 'count') # сколько раз клуб попал в парсинг
).reset_index()

# Размечаем статусы выживаемости
# Если клуб последний раз виделся до финальной даты парсинга -> он закрылся
timeline['is_closed'] = timeline['last_seen'] < max_global_date

# Если клуб впервые появился позже начальной даты -> это новый клуб, а не существовавший изначально
timeline['is_new'] = timeline['first_seen'] > min_global_date

# Считаем известную продолжительность жизни (в днях)
timeline['lifespan_days'] = (timeline['last_seen'] - timeline['first_seen']).dt.days

# --- ШАГ 4: Сбор последних актуальных признаков и расчет дельт ---

# Сортируем исходный df так, чтобы самые свежие записи были последними
df_sorted = df.sort_values(by=['club_key', 'date'])

# 1. Берем последнюю запись для каждого клуба (последние известные характеристики)
latest_features = df_sorted.drop_duplicates(subset=['club_key'], keep='last').copy()

# 2. Берем первую запись для каждого клуба (для расчета разницы со старыми данными)
first_features = df_sorted.drop_duplicates(subset=['club_key'], keep='first').copy()

# Оставляем из первых записей только то, что нужно для сравнения, и переименовываем
cols_for_deltas = ['club_key', 'rating', 'rating_cnt', 'content', 'subs_year', 'subs_month', 'branch_count']
first_features = first_features[cols_for_deltas].rename(columns={
    'rating': 'rating_old',
    'rating_cnt': 'rating_cnt_old',
    'content': 'content_old',
    'subs_year': 'subs_year_old',
    'subs_month': 'subs_month_old',
    'branch_count': 'branch_count_old'
})

# Присоединяем старые значения к последним по club_key
latest_features = latest_features.merge(first_features, on='club_key', how='left')

# --- РАСЧЕТ НОВЫХ ПРИЗНАКОВ ---

# 1. Наличие контактов (True/False)
latest_features['phones_true'] = latest_features['phones'].notna() & (latest_features['phones'] != '')
latest_features['emails_true'] = latest_features['emails'].notna() & (latest_features['emails'] != '')
latest_features['urls_true'] = latest_features['urls'].notna() & (latest_features['urls'] != '')

# 2. Рейтинг: дельта и прирост
valid_rating = latest_features['rating'].notna() & latest_features['rating_old'].notna()
latest_features['rating_delta'] = np.where(valid_rating, latest_features['rating'] - latest_features['rating_old'], np.nan)
latest_features['rating_gain'] = np.where(
    valid_rating & (latest_features['rating_old'] != 0),
    (latest_features['rating'] - latest_features['rating_old']) / latest_features['rating_old'] * 100,
    np.nan
)

# 3. Количество отзывов: дельта и прирост
valid_rating_cnt = latest_features['rating_cnt'].notna() & latest_features['rating_cnt_old'].notna()
latest_features['rating_cnt_delta'] = np.where(valid_rating_cnt, latest_features['rating_cnt'] - latest_features['rating_cnt_old'], np.nan)
latest_features['rating_cnt_gain'] = np.where(
    valid_rating_cnt & (latest_features['rating_cnt_old'] != 0),
    (latest_features['rating_cnt'] - latest_features['rating_cnt_old']) / latest_features['rating_cnt_old'] * 100,
    np.nan
)

# 4. Контент: разница длин списков (по количеству элементов через запятую)
def get_content_len(c):
    if pd.isna(c) or str(c).strip() == '':
        return 0
    return len(str(c).split(','))

content_len_new = latest_features['content'].apply(get_content_len)
content_len_old = latest_features['content_old'].apply(get_content_len)

valid_content = latest_features['content'].notna() & latest_features['content_old'].notna()
latest_features['content_delta'] = np.where(valid_content, content_len_new - content_len_old, np.nan)

# 5. Годовая подписка: дельта и прирост
valid_subs_year = latest_features['subs_year'].notna() & latest_features['subs_year_old'].notna()
latest_features['subs_year_delta'] = np.where(valid_subs_year, latest_features['subs_year'] - latest_features['subs_year_old'], np.nan)
latest_features['subs_year_gain'] = np.where(
    valid_subs_year & (latest_features['subs_year_old'] != 0),
    (latest_features['subs_year'] - latest_features['subs_year_old']) / latest_features['subs_year_old'] * 100,
    np.nan
)

# 6. Месячная подписка: дельта и прирост
valid_subs_month = latest_features['subs_month'].notna() & latest_features['subs_month_old'].notna()
latest_features['subs_month_delta'] = np.where(valid_subs_month, latest_features['subs_month'] - latest_features['subs_month_old'], np.nan)
latest_features['subs_month_gain'] = np.where(
    valid_subs_month & (latest_features['subs_month_old'] != 0),
    (latest_features['subs_month'] - latest_features['subs_month_old']) / latest_features['subs_month_old'] * 100,
    np.nan
)
latest_features['branch_count_delta'] = np.where(valid_subs_month, latest_features['branch_count'] - latest_features['branch_count_old'], np.nan)

# --- Формируем финальный список колонок ---

# Базовые колонки + новые колонки + колонки, в которых нужно было взять "последнее значение"
features_to_keep = [
    'club_key', 'clean_title', 'clean_address', 'clean_district',
    'building_id', 'building_type', 'title', 'region', 'city', 'address', 'lat', 'lon', 
    'district', 'living_area', 'address_comment', 'country', 'district_area', 'settlement',

    # Контакты и новые флаги наличия
    'phones', 'emails', 'urls', 'phones_true', 'emails_true', 'urls_true',

    # Рейтинги и дельты
    'rating', 'rating_delta', 'rating_gain',
    'rating_cnt', 'rating_cnt_delta', 'rating_cnt_gain',

    # Контент
    'content', 'content_delta',

    # Цены
    'subs_year', 'subs_year_delta', 'subs_year_gain',
    'subs_month', 'subs_month_delta', 'subs_month_gain',
    
    # Количество филиалов
    'branch_count_delta', 'branch_count',

    # Атрибуты с последними значениями (из списка в вашем запросе)
    'floors', 'extension', 'accessible_entrance', 'online_training',
    'is_advertised', 'flags', 'temporary_closed', 'photos',
    'created_at', 'updated_at', 'hours_per_day', 'structure_info',
    'material', 'building_floors', 'parking', 'free_parking',
    'payed_parking', 'nearest_station', 'nearest_station_type',
    'nearest_station_d', 'count_stations', 'all_stations',
    'object_floor', 'floor_min', 'floor_max', 'floor_count',
]

# Защита: оставляем только те колонки, которые действительно есть в датафрейме, чтобы избежать KeyError
features_to_keep = [col for col in features_to_keep if col in latest_features.columns]
latest_features = latest_features[features_to_keep]

# Соединяем временную шкалу с фичами
final_aggregated_df = pd.merge(timeline, latest_features, on='club_key', how='left')

print(f"Форма итогового датасета: {final_aggregated_df.shape}")
final_aggregated_df.head()

Форма итогового датасета: (5029, 72)


,club_key,first_seen,last_seen,observations_count,is_closed,is_new,lifespan_days,clean_title,clean_address,clean_district,...,payed_parking,nearest_station,nearest_station_type,nearest_station_d,count_stations,all_stations,object_floor,floor_min,floor_max,floor_count
0,#methodksu_улицамакаренко5ст1а,2024-07-25,2025-02-07,3,True,True,197,#methodksu,улицамакаренко5ст1а,басманный,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,#sekta_3яулицаямскогополя22ст3,2024-02-22,2024-02-22,1,True,False,0,#sekta,3яулицаямскогополя22ст3,беговой,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,#slimfitclub_улицалужники24ст48,2024-02-22,2026-03-24,8,False,False,761,#slimfitclub,улицалужники24ст48,хамовники,...,0.0,Лужники,['mcc'],1400.0,2.0,"{'Лужники': {'distance': 1400, 'comment': 'Мос...",1-2,1.0,2.0,2.0
3,#бойкийфитнес_улицаалександрымонаховой88к3,2024-02-22,2025-11-17,6,True,False,634,#бойкийфитнес,улицаалександрымонаховой88к3,коммунарка,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,#какорех_1йдербеневскийпереулок5,2024-02-22,2024-12-13,3,True,False,295,#какорех,1йдербеневскийпереулок5,даниловский,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [386]:
round(final_aggregated_df[['phones_true', 'emails_true', 'urls_true', 'rating',
       'rating_delta', 'rating_gain', 'rating_cnt', 'rating_cnt_delta',
       'rating_cnt_gain', 'content', 'content_delta', 'subs_year',
       'subs_year_delta', 'subs_year_gain', 'subs_month', 'subs_month_delta',
       'subs_month_gain', 'branch_count', 'branch_count_delta']].describe(),2)

,rating,rating_delta,rating_gain,rating_cnt,rating_cnt_delta,rating_cnt_gain,content_delta,subs_year,subs_year_delta,subs_year_gain,subs_month,subs_month_delta,subs_month_gain,branch_count,branch_count_delta
count,2665.00,2317.00,2309.00,2665.00,2317.00,2317.00,2356.00,655.00,128.00,128.00,1590.00,405.00,405.00,2048.00,113.00
mean,4.52,0.07,3.83,27.95,10.11,114.91,0.40,51645.74,4076.05,20.73,6857.55,472.19,23.27,8.48,-0.08
std,0.83,0.54,29.52,96.14,34.94,747.09,2.23,74113.70,18082.44,83.74,7736.33,4408.73,95.29,18.28,0.36
min,0.00,-2.70,-54.00,1.00,-353.00,-95.12,-16.00,1490.00,-85000.00,-80.95,198.00,-64800.00,-90.00,1.00,-1.00
25%,4.40,0.00,0.00,2.00,0.00,0.00,0.00,18900.00,0.00,0.00,2990.00,0.00,0.00,1.00,0.00
50%,5.00,0.00,0.00,6.00,0.00,0.00,0.00,26500.00,0.00,0.00,4900.00,0.00,0.00,1.00,0.00
75%,5.00,0.00,0.00,21.00,5.00,55.56,1.00,65000.00,1400.00,6.25,8000.00,401.00,11.76,5.00,0.00
max,5.00,5.00,400.00,2368.00,805.00,29800.00,20.00,1100000.00,129700.00,638.92,143373.00,31400.00,1000.00,98.00,2.00


In [396]:
final_aggregated_df.to_excel('final_aggregated_df.xlsx', index = False)

In [395]:
final_aggregated_df['branch_count_delta'].dropna()

9       0.0
12      0.0
31      0.0
119     0.0
122     0.0
       ... 
4665    0.0
4669    0.0
4682    0.0
4787    0.0
4940   -1.0
Name: branch_count_delta, Length: 113, dtype: float64

## Считаем расстояние до центра от каждого фитнеса
- `близость к историческому центру`
- Используем скаченные графы `walk` и `drive` из OSM

In [5]:
import osmnx as ox
import networkx as nx
import pandas as pd
import numpy as np

# Координаты центра
CENTER_LAT = 55.752004
CENTER_LON = 37.617734

def add_network_distances(df, center_lat, center_lon, network_type='drive'):
    print(f"[{network_type}] 1. Скачиваем граф Москвы (это может занять время и RAM)...")
    # Лайфхак: если граф уже скачан, лучше сохранять его на диск через ox.save_graphml 
    # и загружать через ox.load_graphml, чтобы не ждать каждый раз.
    G = ox.graph_from_place('Moscow, Russia', network_type=network_type)
    
    print(f"[{network_type}] 2. Ищем центральную ноду...")
    target_node = ox.distance.nearest_nodes(G, X=center_lon, Y=center_lat)
    
    print(f"[{network_type}] 3. Ищем ближайшие ноды для всех строк датафрейма...")
    # Эта функция векторизована, она быстро отработает для массивов координат
    node_col = f'node_{network_type}'
    df[node_col] = ox.distance.nearest_nodes(G, X=df['lon'].values, Y=df['lat'].values)
    
    print(f"[{network_type}] 4. Считаем расстояния (Single-Target Shortest Path)...")
    # 'length' - это встроенный атрибут osmnx, хранящий длину ребра в метрах.
    # single_target_shortest_path_length учитывает одностороннее движение для машин.
    distances = dict(nx.single_target_shortest_path_length(G, target_node, weight='length'))
    
    print(f"[{network_type}] 5. Привязываем расстояния к датафрейму...")
    dist_col = f'dist_to_center_{network_type}_m'
    
    # Если из какой-то точки физически нельзя доехать до центра (граф разорван), будет NaN
    df[dist_col] = df[node_col].map(lambda x: distances.get(x, np.nan))
    
    # Удаляем техническую колонку с ID ноды
    df = df.drop(columns=[node_col])
    
    return df

# === Использование ===
# Предположим, твой датафрейм называется final_aggregated_df

# Считаем для машин
final_aggregated_df = add_network_distances(final_aggregated_df, CENTER_LAT, CENTER_LON, 'drive')

# Считаем для пешеходов (внимание: пешеходный граф содержит больше тропинок и весит больше!)
final_aggregated_df = add_network_distances(final_aggregated_df, CENTER_LAT, CENTER_LON, 'walk')

[drive] 1. Скачиваем граф Москвы (это может занять время и RAM)...


SSLError: HTTPSConnectionPool(host='overpass-api.de', port=443): Max retries exceeded with url: /api/interpreter (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1016)')))

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict
import re

@dataclass(frozen=True)
class CentralPoint:
    city: str
    label: str
    lat: float
    lon: float

# Координаты в WGS84 (десятичные градусы), приблизительный центр выбранного ориентира.
CENTRAL_POINTS: Dict[str, CentralPoint] = {
    "Москва": CentralPoint("Москва", "Московский Кремль", 55.752004, 37.617734),
    "Санкт-Петербург": CentralPoint("Санкт-Петербург", "Дворцовая площадь", 59.938991, 30.315473),
    "Новосибирск": CentralPoint("Новосибирск", "Площадь Ленина", 55.030148, 82.921283),
    "Екатеринбург": CentralPoint("Екатеринбург", "Площадь 1905 года", 56.837716, 60.596828),
    "Казань": CentralPoint("Казань", "Казанский Кремль", 55.796435, 49.108244),
    "Нижний Новгород": CentralPoint("Нижний Новгород", "Нижегородский кремль", 56.328624, 44.002842),
    "Челябинск": CentralPoint("Челябинск", "Площадь Революции", 55.160200, 61.402670),
    "Самара": CentralPoint("Самара", "Площадь Куйбышева", 53.195490, 50.102143),
    "Омск": CentralPoint("Омск", "Площадь Ленина", 54.982011, 73.378884),
    "Ростов-на-Дону": CentralPoint("Ростов-на-Дону", "Театральная площадь", 47.226778, 39.745223),
    "Уфа": CentralPoint("Уфа", "Площадь Ленина", 54.772878, 56.024106),
    "Красноярск": CentralPoint("Красноярск", "Площадь Революции", 56.010342, 92.852509),
    "Пермь": CentralPoint("Пермь", "Городская эспланада (Театральная площадь)", 58.014722, 56.247222),
    "Воронеж": CentralPoint("Воронеж", "Площадь Ленина", 51.660730, 39.200134),
    "Волгоград": CentralPoint("Волгоград", "Площадь Павших Борцов", 48.707965, 44.515628),
    "Краснодар": CentralPoint("Краснодар", "Главная городская площадь (Театральная)", 45.035451, 38.975376),
}

# Нормализация пользовательского ввода: убираем регистр, пробелы, пунктуацию, общие слова ("г.", "область" и т.п.)
_norm_cleanup_re = re.compile(r"(\bг\.?|\bгород(ской)?|\bобл(асть)?|\bкрай|\bресп(ублика)?|\bокруг)", re.IGNORECASE)
_non_alnum_re = re.compile(r"[^a-zа-я0-9]+")

def _norm(s: str) -> str:
    s = s.strip().lower().replace("ё","е")
    s = _norm_cleanup_re.sub("", s)
    s = _non_alnum_re.sub("", s)
    return s

# Ручные алиасы (основные разговорные формы и варианты написания регионов, которые должны сводиться к городу-миллионнику).
_alias_pairs = [
    ("мск", "Москва"),
    ("moscow", "Москва"),
    ("moskva", "Москва"),
    ("спб", "Санкт-Петербург"),
    ("spb", "Санкт-Петербург"),
    ("питер", "Санкт-Петербург"),
    ("ленинград", "Санкт-Петербург"),
    ("новосиб", "Новосибирск"),
    ("nsk", "Новосибирск"),
    ("екб", "Екатеринбург"),
    ("екат", "Екатеринбург"),
    ("свердловск", "Екатеринбург"),
    ("татарстан", "Казань"),
    ("республика татарстан", "Казань"),
    ("нижний", "Нижний Новгород"),
    ("горький", "Нижний Новгород"),
    ("челяба", "Челябинск"),
    ("куйбышев", "Самара"),
    ("ростов", "Ростов-на-Дону"),   # в контексте миллионников
    ("башкортостан", "Уфа"),
    ("республика башкортостан", "Уфа"),
    ("красноярский край", "Красноярск"),
    ("пермский край", "Пермь"),
    ("сталинград", "Волгоград"),
    ("краснодарский край", "Краснодар"),
]

_ALIASES = { _norm(k): v for k, v in _alias_pairs }
_ALIASES.update({ _norm(k): k for k in CENTRAL_POINTS })  # сами канонические имена тоже

def get_central_point(city: str) -> CentralPoint:
    """Вернуть тематический «центр города» (ориентир + координаты) для города-миллионника России.

    Принимает русское название города или распространённое сокращение / название субъекта (например, "Башкортостан").
    Бросает ValueError, если город не распознан.
    """
    key = _norm(city)
    canon = _ALIASES.get(key)
    if canon is None:
        # Попробуем простое подстроковое совпадение
        for k, v in _ALIASES.items():
            if k in key or key in k:
                canon = v
                break
    if canon is None:
        valid = ", ".join(CENTRAL_POINTS)
        raise ValueError(f"Неизвестный город: {city!r}. Допустимые города: {valid}")
    return CENTRAL_POINTS[canon]

city_name = "Казань"
cp = get_central_point(city_name)
latitude = cp.lat
longitude = cp.lon
print(f"latitude: {latitude}, longitude: {longitude}")

In [ ]:
dentistry_directory = r"D:\Users\Mariia\Desktop\FitnessData\Кейсы\Стоматологии\центр-периферия\data\dentistry"
builds_directory    = r"D:\Users\Mariia\Desktop\FitnessData\Кейсы\Стоматологии\центр-периферия\data\reforma"

def clinics_within_iso(distance, clinics, lat, lon):
    center_point = (lat, lon)
    
    graph = ox.graph.graph_from_point(
        center_point,
        dist=distance,
        dist_type='network', network_type='walk',
        simplify=True
    )
    
    G = ox.project_graph(graph, to_crs="EPSG:32637")
    # Use this line if the coordinates sistem returned from polys is changed from the original (check which crs you are using)
    
    gdf_nodes = ox.graph_to_gdfs(G, edges=False)
    x, y = gdf_nodes['geometry'].union_all().centroid.xy
    center_node = ox.nearest_nodes(G, Y = y[0], X= x[0])
    subgraph = nx.ego_graph(G, center_node, radius=distance, distance='length')
    node_points = [Point(data['x'], data['y']) for node, data in subgraph.nodes(data=True)]
    polys = gpd.GeoSeries(node_points).union_all().convex_hull
    
    # сохраняем в GeoDataFrame
    iso_dict = {'iso': ['n min'], 'geometry': polys}
    gdf_iso = gpd.GeoDataFrame(iso_dict, crs="EPSG:32637").to_crs(proj_crs)
    
    # Пространственная выборка - оставляем только точки кафешек в изохроне
    points_clinics = clinics.sjoin(gdf_iso,  how='inner', predicate='intersects')

    return len(points_clinics)/len(clinics)



cities = [
    "Москва", "Санкт-Петербург", "Новосибирск", "Екатеринбург", "Казань",
    "Нижний Новгород", "Красноярск",
    "Челябинск", "Уфа", "Самара",
    "Ростов-на-Дону", "Омск", "Волгоград", "Воронеж", "Пермь",
    "Краснодар",
][::-1]

# stats = {'city': [], 'center_lat': [], 'center_lon': [], 'place': [],
#          '15 min_new_p': [], '20 min_new_p': [], '30 min_new_p': [], '40 min_new_p': [],
#          '15 min_all_p': [], '20 min_all_p': [], '30 min_all_p': [], '40 min_all_p': [],
#          'new_clinics_count': [], 'all_clinics_count': []}

d = [1200, 1600, 2400, 3200]
min_new = ['15 min_new_p', '20 min_new_p', '30 min_new_p', '40 min_new_p']
min_all = ['15 min_all_p', '20 min_all_p', '30 min_all_p', '40 min_all_p']
proj_crs = "EPSG:32637"

for city in cities:
    start_time = time.time()  # Начало отсчета времени
    print(f"\n— {city}")
    
    # # дома
    # builds_path = os.path.join(builds_directory, city, "builds.gpkg")
    # builds = gpd.read_file(builds_path).to_crs(proj_crs)

    # все стоматологии
    all_path = os.path.join(dentistry_directory, city, "all_city_dentistries.gpkg")
    all_clinics = gpd.read_file(all_path).to_crs(proj_crs)
    
    # новые стоматологии
    new_path = os.path.join(dentistry_directory, city, "new_city_dentistries.gpkg")
    new_clinics = gpd.read_file(new_path).to_crs(proj_crs)

    # ______  тут основной код цикла
    cp = get_central_point(city)
    latitude = cp.lat
    longitude = cp.lon
    
    stats['city'].append(city)
    stats['center_lat'].append(latitude)
    stats['center_lon'].append(longitude)
    stats['place'].append(cp.label)
    stats['all_clinics_count'].append(len(all_clinics))
    stats['new_clinics_count'].append(len(new_clinics))
    
    for i in range(len(d)): # 15, 20, 30, 40 min   
        clinics_counts = clinics_within_iso(d[i], new_clinics, latitude, longitude)
        stats[min_new[i]].append(clinics_counts)

    for i in range(len(d)): # 15, 20, 30, 40 min   
        clinics_counts = clinics_within_iso(d[i], all_clinics, latitude, longitude)
        stats[min_all[i]].append(clinics_counts)

    print(f'Доля всех клиник в 20-ти минутах пешей доступности от центра: {stats['20 min_all_p'][-1]*100:.2f} %')
    print(f'Доля новых клиник в 20-ти минутах пешей доступности от центра: {stats['20 min_new_p'][-1]*100:.2f} %')
    # _______ конец основного кода
    
    # # сохранение
    # out_path_all_clinics = os.path.join(dentistry_directory, city, "all_city_dentistries_iso_dist.gpkg")
    # all_clinics.to_file(out_path_all_clinics, driver="GPKG", index=False)

    # out_path_new_clinics = os.path.join(dentistry_directory, city, "all_city_dentistries_iso_dist.gpkg")
    # all_clinics.to_file(out_path_all_clinics, driver="GPKG", index=False)
    
    # out_path_builds = os.path.join(builds_directory, city, "builds_iso_dist.gpkg")
    # builds_merged.to_file(out_path_builds, driver="GPKG", index=False)
    
    end_time = time.time()
    execution_time = end_time - start_time
    
    print(f"Время выполнения функции: {execution_time/60:.4f} минут")
    print(f"{(execution_time/8):.4f} сек на точку")
    
iso_center_stats = pd.DataFrame(stats)
os.makedirs(r"data\calculates", exist_ok=True)
out_excel = r"data\calculates\iso_center.xlsx"
iso_center_stats.to_excel(out_excel, index=False)

print("\nСводная таблица сохранена:", out_excel)

In [ ]:
gdf_points = gpd.GeoDataFrame(
    df_concat,
    geometry=gpd.points_from_xy(df_concat['lon'], df_concat['lat']),
    crs="EPSG:4326"
)

moscow_boundary = gpd.read_file('moscow_boundary.gpkg')

if moscow_boundary.crs != gdf_points.crs:
    moscow_boundary = moscow_boundary.to_crs(gdf_points.crs)

gdf_clipped = gpd.clip(gdf_points, moscow_boundary)

gdf_clipped['date'].value_counts()

In [ ]:
import osmnx as ox
import networkx as nx
import pandas as pd
import numpy as np

# Координаты центра
CENTER_LAT = 55.752004
CENTER_LON = 37.617734

def add_network_distances(df, center_lat, center_lon, network_type='drive'):
    print(f"[{network_type}] 1. Скачиваем граф Москвы (это может занять время и RAM)...")
    # Лайфхак: если граф уже скачан, лучше сохранять его на диск через ox.save_graphml 
    # и загружать через ox.load_graphml, чтобы не ждать каждый раз.
    G = ox.graph_from_place('Moscow, Russia', network_type=network_type)
    
    print(f"[{network_type}] 2. Ищем центральную ноду...")
    target_node = ox.distance.nearest_nodes(G, X=center_lon, Y=center_lat)
    
    print(f"[{network_type}] 3. Ищем ближайшие ноды для всех строк датафрейма...")
    # Эта функция векторизована, она быстро отработает для массивов координат
    node_col = f'node_{network_type}'
    df[node_col] = ox.distance.nearest_nodes(G, X=df['lon'].values, Y=df['lat'].values)
    
    print(f"[{network_type}] 4. Считаем расстояния (Single-Target Shortest Path)...")
    # 'length' - это встроенный атрибут osmnx, хранящий длину ребра в метрах.
    # single_target_shortest_path_length учитывает одностороннее движение для машин.
    distances = dict(nx.single_target_shortest_path_length(G, target_node, weight='length'))
    
    print(f"[{network_type}] 5. Привязываем расстояния к датафрейму...")
    dist_col = f'dist_to_center_{network_type}_m'
    
    # Если из какой-то точки физически нельзя доехать до центра (граф разорван), будет NaN
    df[dist_col] = df[node_col].map(lambda x: distances.get(x, np.nan))
    
    # Удаляем техническую колонку с ID ноды
    df = df.drop(columns=[node_col])
    
    return df

# === Использование ===
# Предположим, твой датафрейм называется final_aggregated_df

# Считаем для машин
# final_aggregated_df = add_network_distances(final_aggregated_df, CENTER_LAT, CENTER_LON, 'drive')

# Считаем для пешеходов (внимание: пешеходный граф содержит больше тропинок и весит больше!)
# final_aggregated_df = add_network_distances(final_aggregated_df, CENTER_LAT, CENTER_LON, 'walk')

# print(final_aggregated_df.head())


In [8]:
# Координаты центра
CENTER_LAT = 55.752004
CENTER_LON = 37.617734

# Путь к папке с графами
GRAPH_PATH = r'D:/Users/Mariia/Desktop/Курсач/4 курс/2gis'

def add_network_distances(df, center_lat, center_lon, network_type='drive'):
    print(f"[{network_type}] 1. Загружаем граф с диска...")
    
    # Формируем путь к файлу
    graph_file = f'{GRAPH_PATH}/moscow_{network_type}.graphml'
    G = ox.load_graphml(graph_file)
    
    print(f"[{network_type}] 2. Ищем центральную ноду...")
    target_node = ox.distance.nearest_nodes(G, X=center_lon, Y=center_lat)
    
    print(f"[{network_type}] 3. Ищем ближайшие ноды для всех строк датафрейма...")
    node_col = f'node_{network_type}'
    df[node_col] = ox.distance.nearest_nodes(G, X=df['lon'].values, Y=df['lat'].values)
    
    print(f"[{network_type}] 4. Считаем расстояния (Single-Target Shortest Path)...")
    # Важно: при загрузке из graphml типы данных могут измениться, 
    # поэтому убедимся, что вес 'length' числовой
    for u, v, data in G.edges(data=True):
        if 'length' in data:
            data['length'] = float(data['length'])
    
    distances = dict(nx.single_source_dijkstra_path_length(G, target_node, weight='length'))
    
    print(f"[{network_type}] 5. Привязываем расстояния к датафрейму...")
    dist_col = f'dist_to_center_{network_type}_m'
    df[dist_col] = df[node_col].map(lambda x: distances.get(x, np.nan))
    
    # Удаляем техническую колонку
    df = df.drop(columns=[node_col])
    
    return df

print("Функция готова к использованию!")
print("\nПример вызова:")
print("final_aggregated_df = add_network_distances(final_aggregated_df, CENTER_LAT, CENTER_LON, 'drive')")
print("final_aggregated_df = add_network_distances(final_aggregated_df, CENTER_LAT, CENTER_LON, 'walk')")
final_aggregated_df = pd.read_excel('final_aggregated_df.xlsx')
print(final_aggregated_df.shape)

final_aggregated_df = add_network_distances(final_aggregated_df, CENTER_LAT, CENTER_LON, 'drive')

# Считаем для пешеходов (внимание: пешеходный граф содержит больше тропинок и весит больше!)
final_aggregated_df = add_network_distances(final_aggregated_df, CENTER_LAT, CENTER_LON, 'walk')

Функция готова к использованию!

Пример вызова:
final_aggregated_df = add_network_distances(final_aggregated_df, CENTER_LAT, CENTER_LON, 'drive')
final_aggregated_df = add_network_distances(final_aggregated_df, CENTER_LAT, CENTER_LON, 'walk')
(5029, 72)
[drive] 1. Загружаем граф с диска...
[drive] 2. Ищем центральную ноду...
[drive] 3. Ищем ближайшие ноды для всех строк датафрейма...
[drive] 4. Считаем расстояния (Single-Target Shortest Path)...
[drive] 5. Привязываем расстояния к датафрейму...
[walk] 1. Загружаем граф с диска...
[walk] 2. Ищем центральную ноду...
[walk] 3. Ищем ближайшие ноды для всех строк датафрейма...
[walk] 4. Считаем расстояния (Single-Target Shortest Path)...
[walk] 5. Привязываем расстояния к датафрейму...


In [10]:
final_aggregated_df[['dist_to_center_drive_m','dist_to_center_walk_m']].describe()

,dist_to_center_drive_m,dist_to_center_walk_m
count,5016.000000,5029.000000
mean,14164.061478,13528.754905
std,8119.004913,7780.127454
min,533.665077,927.907071
25%,8047.548132,7584.218801
50%,13297.221796,12749.695816
75%,18059.953294,17654.578416
max,78711.518622,76562.051758


In [12]:
final_aggregated_df.to_excel('fitness_w_centerd.xlsx', index=False)
final_aggregated_df.to_csv('fitness_w_centerd.csv', index=False)

## Конкуренты

Исправляем последний статус (открыт или закрыт) фитнес-объектов из ТиНАО и Зеленограда, так как многие не попали в последнюю выгрузку

In [185]:
fitness = pd.read_csv('fitness_w_centerd.csv')
lst = ['коммунарка','филимонковский', 'краснопахорский',
       'щербинка', 'троицк', 'крюково', 'савелки', 'внуково',
      'силино', 'матушкино', 'староекрюково', 'вороново', 'бекасово',
      'южноетушино', 'северноетушино', 'капотня']

new_mosc_all = fitness[(fitness['clean_district'].isin(lst)) & (fitness['is_closed']==True)]
print(len(new_mosc_all))
# new_mosc_all.to_excel('new_mosc.xlsx')

464


In [208]:
fitness = pd.read_csv('fitness_w_centerd.csv')
lst = ['коммунарка','филимонковский', 'краснопахорский',
       'щербинка', 'троицк', 'крюково', 'савелки', 'внуково',
      'силино', 'матушкино', 'староекрюково', 'вороново', 'бекасово',
      'южноетушино', 'северноетушино', 'капотня']

new_mosc_all = fitness[~(fitness['clean_district'].isin(lst)) & (fitness['is_closed']==True)]
print(len(new_mosc_all))
new_mosc_all.to_excel('new_mosc222.xlsx')

2774


Далее проверем вручную статусы объектов

In [209]:
# 0. Загружаем данные
new_mosc = pd.read_excel('new_mosc.xlsx')
#fitness = pd.read_csv('fitness_w_centerd.csv')
fitness = fitness.reset_index().rename(columns={'index':'fitness_id'})
fitness['fitness_id']=fitness['fitness_id']+1

fitness['first_seen'] = pd.to_datetime(fitness['first_seen'], errors='coerce')
fitness['last_seen'] = pd.to_datetime(fitness['last_seen'], errors='coerce')

# 1. Получаем список fitness_id, у которых status == 1
target_ids = new_mosc.loc[new_mosc['status'] == 1, 'fitness_id'].tolist()

# 2. Создаем маску для фильтрации датафрейма fitness
mask = fitness['fitness_id'].isin(target_ids)

# 3. Вносим изменения по маске
# Увеличиваем счетчик наблюдений на 1
fitness.loc[mask, 'observations_count'] = fitness.loc[mask, 'observations_count'] + 1

# Меняем статус на "не закрыт"
fitness.loc[mask, 'is_closed'] = False

# Обновляем дату последнего наблюдения
fitness.loc[mask, 'last_seen'] = '2026-03-24'

# Увеличиваем время жизни на 35 дней
fitness.loc[mask, 'lifespan_days'] = (fitness.loc[mask, 'last_seen'] - fitness.loc[mask, 'first_seen']).dt.days
# fitness = fitness[fitness.columns[1:]]

In [215]:
# 1 - объект все-таки открыт
# 0 - объект закрыт
print(len(new_mosc))
print()
print(new_mosc['status'].value_counts())
print()
print(round(new_mosc['status'].value_counts()/len(new_mosc),2))

3235

status
1.0    1710
0.0    1520
Name: count, dtype: int64

status
1.0    0.53
0.0    0.47
Name: count, dtype: float64


In [217]:
fitness['is_closed'].value_counts()

is_closed
False    3501
True     1528
Name: count, dtype: int64

In [218]:
fitness.to_excel('fitness_aggregated_cleared.xlsx',index=False)
fitness.to_csv('fitness_aggregated_cleared.csv',index=False)

### Пешеходная изохрона 15 мин

In [9]:
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning, message='Mean of empty slice')

# --- НАСТРОЙКИ ---
GRAPH_PATH = r'D:/Users/Mariia/Desktop/Курсач/4 курс/2gis' 
WALK_SPEED_KMH = 4.5
METERS_PER_MIN = WALK_SPEED_KMH * 1000 / 60
WALK_TIME_MIN = 15

# ... (Здесь остаются твои AGG_RULES и ALL_EMPTY_METRICS) ...

# --- ЗАГРУЗКА ГРАФА ---
print("Загружаем пешеходный граф Москвы...")
# G_walk = ox.load_graphml(f'{GRAPH_PATH}/moscow_walk.graphml')
# G_walk = ox.project_graph(G_walk) 
current_crs = G_walk.graph['crs']

for u, v, k, edge_data in G_walk.edges(keys=True, data=True):
    length = edge_data.get('length')
    if length is not None:
        edge_data['time'] = float(length) / METERS_PER_MIN

# --- ПОДГОТОВКА ДАННЫХ ---
print("Подготовка пространственных данных...")
# data['geometry'] = gpd.points_from_xy(data.lon, data.lat)
# gdf_data = gpd.GeoDataFrame(data, geometry='geometry', crs="EPSG:4326").to_crs(current_crs)

print('НАЧАЛО ОБРАБОТКИ')
# fitness = pd.read_csv('fitness_w_centerd.csv')

# --- NEW: ПОДГОТОВКА ГЕОДАТАФРЕЙМА ФИТНЕСОВ ---
# Создаем геометрию для самих фитнесов, чтобы искать их внутри изохрон
fitness['geometry'] = gpd.points_from_xy(fitness.lon, fitness.lat)
gdf_fitness = gpd.GeoDataFrame(fitness, geometry='geometry', crs="EPSG:4326")
gdf_fitness = gdf_fitness.to_crs(current_crs) # Проецируем в систему координат графа

results = []
print(f"Начинаем обработку {len(fitness)} объектов...")

start = time.time()

for idx, row in fitness.iterrows():
    lat, lon = row['lat'], row['lon']
    
    if idx % 250 == 0:
        print(f"Обрабатываем объект {idx+1}/{len(fitness)}...")
    
    try:
        # --- ПОСТРОЕНИЕ ИЗОХРОНЫ ---
        point_wgs84 = Point(lon, lat)
        point_graph_crs = gpd.GeoSeries([point_wgs84], crs="EPSG:4326").to_crs(current_crs).iloc[0]
        
        center_node = ox.distance.nearest_nodes(G_walk, X=point_graph_crs.x, Y=point_graph_crs.y)
        subgraph = nx.ego_graph(G_walk, center_node, radius=WALK_TIME_MIN, distance='time')
        
        if len(subgraph.nodes()) < 3:
            # NEW: Добавляем пустой список конкурентов, если изохрона не построилась
            results.append({'fitness_id': idx, 'competitors_list': [], **ALL_EMPTY_METRICS})
            continue

        poly = gpd.GeoSeries([Point(node_data['x'], node_data['y']) for _, node_data in subgraph.nodes(data=True)]).unary_union.convex_hull
        poly_gdf = gpd.GeoDataFrame(geometry=[poly], crs=current_crs)

        # --- NEW: ПОИСК КОНКУРЕНТОВ В ИЗОХРОНЕ ---
        # Пересекаем все фитнесы (gdf_fitness) с текущей изохроной (poly_gdf)
        competitors_hits = gpd.sjoin(gdf_fitness, poly_gdf, how='inner', predicate='intersects')
        
        # Получаем список индексов фитнесов, попавших в полигон
        competitor_ids = competitors_hits.index.tolist()
        
        # Обязательно исключаем сам текущий объект из списка его конкурентов
        if idx in competitor_ids:
            competitor_ids.remove(idx)

        # --- ПРОСТРАНСТВЕННОЕ СОЕДИНЕНИЕ С ДРУГИМИ ДАННЫМИ (data) ---
        # hits = gpd.sjoin(poly_gdf, gdf_data, how='inner', predicate='intersects')
        # if hits.empty:
        #     results.append({'fitness_id': idx, 'competitors_list': competitor_ids, **ALL_EMPTY_METRICS})
        #     continue

        # --- РАСЧЕТ МЕТРИК ---
        row_result = {
            'fitness_id': idx,
            'competitors_list': competitor_ids # NEW: сохраняем список id конкурентов
        }
        
        # ... (Здесь остается твой цикл for prefix, config in AGG_RULES.items():) ...

        results.append(row_result)
        
    except Exception as e:
        print(f"Ошибка на индексе {idx}: {e}")
        # NEW: не забываем добавить пустой список при ошибке
        results.append({'fitness_id': idx, 'competitors_list': [], **ALL_EMPTY_METRICS})

# --- СОХРАНЕНИЕ РЕЗУЛЬТАТА ---
res_df = pd.DataFrame(results)
print(f"Обработка завершена за {time.time() - start:.2f} секунд.")
res_df.head()

Загружаем пешеходный граф Москвы...
Подготовка пространственных данных...
НАЧАЛО ОБРАБОТКИ
Начинаем обработку 5029 объектов...
Обрабатываем объект 1/5029...
Обрабатываем объект 251/5029...
Обрабатываем объект 501/5029...
Обрабатываем объект 751/5029...
Обрабатываем объект 1001/5029...
Обрабатываем объект 1251/5029...
Обрабатываем объект 1501/5029...
Обрабатываем объект 1751/5029...
Обрабатываем объект 2001/5029...
Обрабатываем объект 2251/5029...
Обрабатываем объект 2501/5029...
Обрабатываем объект 2751/5029...
Обрабатываем объект 3001/5029...
Обрабатываем объект 3251/5029...
Обрабатываем объект 3501/5029...
Обрабатываем объект 3751/5029...
Обрабатываем объект 4001/5029...
Обрабатываем объект 4251/5029...
Обрабатываем объект 4501/5029...
Обрабатываем объект 4751/5029...
Обрабатываем объект 5001/5029...
Обработка завершена за 17898.38 секунд.


,fitness_id,competitors_list
0,0,"[44, 45, 255, 472, 473, 627, 790, 795, 838, 11..."
1,1,"[32, 37, 70, 585, 714, 1097, 1232, 1234, 1235,..."
2,2,"[529, 536, 1229, 2550, 3144, 3209, 3453, 3485,..."
3,3,"[95, 179, 221, 222, 265, 798, 953, 968, 1310, ..."
4,4,"[490, 652, 3404, 4492]"


### Обрабатываем конкурентов и объекты своей же сети по временному промежутку

In [219]:
res_df.columns = ['fitness_id', 'competitors_list', 'competitors_count',
       'competitors_during_all_comp',
       'competitors_during_part_comp',
       'competitors_during_all_comp_count',
       'competitors_during_part_comp_count',
       'competitors_during_all_cann',
       'competitors_during_part_cann',
       'competitors_during_all_cann_count',
       'competitors_during_part_cann_count',
       'competitors_comp_cnt', 'competitors_cann_cnt']

In [223]:
# Убираем найденные вручную дубли
numbers_to_remove = [1479, 508, 509, 510, 2156, 2386, 3827, 4883,  2702,
               84, 223,1118, 2041,2948, 3819,93,143, 209, 478, 552,
               774, 861, 1074, 1170,1292, 1581, 1879, 1893,
                 1967, 1974,2385,2946,4492]] 

numbers_to_remove = set([num for pair in pairs for num in pair] + ids_to_drop)

columns_to_clean = [
    'competitors_during_all_cann',
    'competitors_during_part_cann',
    'competitors_list',
    'competitors_during_all_comp',
    'competitors_during_part_comp'
]

# 3. Функция фильтрации
def filter_ids(val):
    # Проверяем, что значение действительно является списком (защита от NaN/float)
    if isinstance(val, list):
        return [x for x in val if x not in numbers_to_remove]
    return val

# 4. Применяем функцию к нужным колонкам
for col in columns_to_clean:
    res_df[col] = res_df[col].apply(filter_ids)

In [224]:
res_df['competitors_count'] = res_df['competitors_list'].apply(lambda x: len(x) if isinstance(x, list) else 0)

# Переводим в даты (на всякий случай используем coerce, чтобы кривые даты стали NaT, а не роняли код)
fitness['first_seen'] = pd.to_datetime(fitness['first_seen'], errors='coerce')
fitness['last_seen'] = pd.to_datetime(fitness['last_seen'], errors='coerce')

# Создаем словари
first_seen_dict = fitness.set_index('fitness_id')['first_seen'].to_dict()
last_seen_dict = fitness.set_index('fitness_id')['last_seen'].to_dict()
title_dict = fitness.set_index('fitness_id')['clean_title'].to_dict()

# Переписываем функцию
def to_def_true_comp(row):
    during_all_comp = []
    during_part_comp = []
    during_all_cann = []
    during_part_cann = []
    
    # Достаем даты самого объекта
    my_first = first_seen_dict.get(row['fitness_id'])
    my_last = last_seen_dict.get(row['fitness_id'])
    my_title = title_dict.get(row['fitness_id'])
    
    # --- ЗАЩИТА 1: Проверяем самого себя! ---
    # Если у нас нет дат в словаре, мы не можем сравнивать (< или >).
    # Сразу возвращаем 8 пустых значений для 8 новых колонок.
    if pd.isna(my_first) or pd.isna(my_last) or pd.isna(my_title):
        return pd.Series([[], [], 0, 0, [], [], 0, 0])
    
    # --- ЗАЩИТА 2: Проверяем список конкурентов ---
    if not isinstance(row['competitors_list'], list):
        return pd.Series([[], [], 0, 0, [], [], 0, 0])

    for num in row['competitors_list']:
        comp_first = first_seen_dict.get(num)
        comp_last = last_seen_dict.get(num)
        title = title_dict.get(num)
        
        # Защита конкурента: пропускаем, если у него нет дат
        if pd.isna(comp_first) or pd.isna(comp_last) or pd.isna(title):
            continue
            
        # Теперь мы безопасно сравниваем даты (Timestamp)
        if comp_first <= my_first and comp_last >= my_last and title != my_title:
            during_all_comp.append(num)
        elif comp_first <= my_first and comp_last >= my_last and title == my_title:
            during_all_cann.append(num)
        elif comp_last >= my_last and title != my_title:
            during_part_comp.append(num)
        elif comp_last >= my_last and title == my_title:
            during_part_cann.append(num)
            
    # Возвращаем ровно 8 элементов
    return pd.Series([
        during_all_comp, 
        during_part_comp, 
        len(during_all_comp), 
        len(during_part_comp),
        during_all_cann, 
        during_part_cann, 
        len(during_all_cann), 
        len(during_part_cann)
    ])

# Применяем функцию
res_df[['competitors_during_all_comp', 'competitors_during_part_comp', 
        'competitors_during_all_comp_count', 'competitors_during_part_comp_count',
       'competitors_during_all_cann', 'competitors_during_part_cann', 
        'competitors_during_all_cann_count', 'competitors_during_part_cann_count']] = res_df.apply(to_def_true_comp, axis=1)

# Считаем доли, добавляем fillna(0) от деления на ноль
res_df['competitors_comp_cnt'] = (res_df['competitors_during_all_comp_count'] / res_df['competitors_count']).fillna(0)
res_df['competitors_cann_cnt'] = (res_df['competitors_during_all_cann_count'] / res_df['competitors_count']).fillna(0)

# Выводим результат
res_df.sample(10)

,fitness_id,competitors_list,competitors_count,competitors_during_all_comp,competitors_during_part_comp,competitors_during_all_comp_count,competitors_during_part_comp_count,competitors_during_all_cann,competitors_during_part_cann,competitors_during_all_cann_count,competitors_during_part_cann_count,competitors_comp_cnt,competitors_cann_cnt
1511,1511,"[10, 172, 405, 407, 1414, 1495, 1551, 1583, 15...",27,"[405, 1495, 1608, 1633, 1648, 1806, 2101, 2126...","[10, 172, 1414, 1551, 1583, 1592, 1611, 2097, ...",11,12,[],[],0,0,0.407407,0.0
274,274,"[943, 1445, 1446, 1605, 1700, 2409, 3235, 3883...",12,"[943, 1605, 1700, 2409, 3235, 3883, 3911, 4472...","[1445, 1446]",10,2,[],[],0,0,0.833333,0.0
4705,4705,"[505, 531, 683, 796, 868, 978, 980, 1055, 1057...",44,"[505, 531, 683, 868, 978, 980, 1055, 1193, 152...","[796, 1057, 1125, 1521, 1944, 2907, 3519, 4353]",33,8,[],[],0,0,0.750000,0.0
149,149,"[100, 780, 1023, 1113, 1538, 2035, 2206, 2640]",8,"[2035, 2206, 2640]","[780, 1023, 1538]",3,3,[],[],0,0,0.375000,0.0
1469,1469,"[546, 2048]",2,[],[546],0,1,[],[],0,0,0.000000,0.0
2564,2564,"[62, 63, 1539]",3,"[62, 63]",[1539],2,1,[],[],0,0,0.666667,0.0
3746,3746,"[30, 31, 61, 2367, 2565, 2662, 2663, 2664, 293...",19,"[30, 31, 2367, 2664]","[61, 2565, 2662, 2663, 2935, 2983, 3412, 4111,...",4,15,[],[],0,0,0.210526,0.0
2761,2761,"[348, 380, 547, 550, 551, 766, 842, 921, 947, ...",26,"[348, 380, 766, 842, 921, 1161, 1162, 1740, 18...","[947, 1759, 4286]",15,3,[],[],0,0,0.576923,0.0
37,37,"[1, 32, 70, 585, 714, 1097, 1232, 1234, 1235, ...",32,"[714, 1234, 1608, 1655, 1806, 1816, 2101, 2126...","[1, 32, 70, 585, 1097, 1235, 1495, 1551, 1583,...",11,19,[],[],0,0,0.343750,0.0
2387,2387,"[114, 160, 228, 245, 395, 695, 739, 875, 952, ...",24,"[114, 160, 395, 695, 1107, 1558, 2250, 2298, 2...",[2973],14,1,[],[],0,0,0.583333,0.0


In [225]:
res_df.columns = ['fitness_id', 'competitors_list_15ped', 'competitors_count_15ped',
       'competitors_during_all_comp_15ped', 'competitors_during_part_comp_15ped',
       'competitors_during_all_comp_count_15ped',
       'competitors_during_part_comp_count_15ped', 'competitors_during_all_cann_15ped',
       'competitors_during_part_cann_15ped', 'competitors_during_all_cann_count_15ped',
       'competitors_during_part_cann_count_15ped', 'competitors_comp_cnt_15ped',
       'competitors_cann_cnt_15ped']

res_df.to_excel('fitness_competitors_15ped.xlsx', index=False)
res_df.to_csv('fitness_competitors_15ped.csv', index=False)

In [226]:
res_df['competitors_during_all_cann_count_15ped'].value_counts()

competitors_during_all_cann_count_15ped
0    4796
1     198
2      30
3       4
4       1
Name: count, dtype: int64

### Автомобильная изохрона 15 мин

In [13]:
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning, message='Mean of empty slice')

# --- НАСТРОЙКИ ---
GRAPH_PATH = r'D:/Users/Mariia/Desktop/Курсач/4 курс/2gis' 
WALK_SPEED_KMH = 4.5
METERS_PER_MIN = WALK_SPEED_KMH * 1000 / 60
WALK_TIME_MIN = 15


DRIVE_SPEED_KMH = 30 # Средняя скорость авто в городе
METERS_PER_MIN_DRIVE = DRIVE_SPEED_KMH * 1000 / 60
DRIVE_TIME_MIN = 15
iso_type = 'drive'
MAX_DRIVE_METERS = DRIVE_TIME_MIN * METERS_PER_MIN_DRIVE

# --- ЗАГРУЗКА ГРАФА ---
print("Загружаем пешеходный граф Москвы...")
G_drive = ox.load_graphml(f'{GRAPH_PATH}/moscow_drive.graphml')
G_drive = ox.project_graph(G_drive) # Проекция в метры (UTM)
current_crs = G_walk.graph['crs']

for u, v, k, edge_data in G_walk.edges(keys=True, data=True):
    length = edge_data.get('length')
    if length is not None:
        edge_data['time'] = float(length) / METERS_PER_MIN_DRIVE

# --- ПОДГОТОВКА ДАННЫХ ---
print("Подготовка пространственных данных...")
# data['geometry'] = gpd.points_from_xy(data.lon, data.lat)
# gdf_data = gpd.GeoDataFrame(data, geometry='geometry', crs="EPSG:4326").to_crs(current_crs)

print('НАЧАЛО ОБРАБОТКИ')
# fitness = pd.read_csv('fitness_w_centerd.csv')

# ПОДГОТОВКА ГЕОДАТАФРЕЙМА ФИТНЕСОВ ---
# Создаем геометрию для самих фитнесов, чтобы искать их внутри изохрон
fitness['geometry'] = gpd.points_from_xy(fitness.lon, fitness.lat)
gdf_fitness = gpd.GeoDataFrame(fitness, geometry='geometry', crs="EPSG:4326")
gdf_fitness = gdf_fitness.to_crs(current_crs) # Проецируем в систему координат графа

results = []
print(f"Начинаем обработку {len(fitness)} объектов...")

start = time.time()

for idx, row in fitness.iterrows():
    lat, lon = row['lat'], row['lon']
    
    if idx % 250 == 0:
        print(f"Обрабатываем объект {idx+1}/{len(fitness)}...")
    
    try:
        # --- ПОСТРОЕНИЕ ИЗОХРОНЫ ---
        point_wgs84 = Point(lon, lat)
        point_graph_crs = gpd.GeoSeries([point_wgs84], crs="EPSG:4326").to_crs(current_crs).iloc[0]
        
        center_node = ox.distance.nearest_nodes(G_drive, X=point_graph_crs.x, Y=point_graph_crs.y)
        subgraph = nx.ego_graph(G_drive, center_node, radius=DRIVE_TIME_MIN, distance='time')
        
        if len(subgraph.nodes()) < 3:
            # NEW: Добавляем пустой список конкурентов, если изохрона не построилась
            results.append({'fitness_id': idx, 'competitors_list': [], **ALL_EMPTY_METRICS})
            continue

        poly = gpd.GeoSeries([Point(node_data['x'], node_data['y']) for _, node_data in subgraph.nodes(data=True)]).unary_union.convex_hull
        poly_gdf = gpd.GeoDataFrame(geometry=[poly], crs=current_crs)

        # ПОИСК КОНКУРЕНТОВ В ИЗОХРОНЕ ---
        # Пересекаем все фитнесы (gdf_fitness) с текущей изохроной (poly_gdf)
        competitors_hits = gpd.sjoin(gdf_fitness, poly_gdf, how='inner', predicate='intersects')
        
        # Получаем список индексов фитнесов, попавших в полигон
        competitor_ids = competitors_hits.index.tolist()
        
        # Обязательно исключаем сам текущий объект из списка его конкурентов
        if idx in competitor_ids:
            competitor_ids.remove(idx)

        # --- РАСЧЕТ МЕТРИК ---
        row_result = {
            'fitness_id': idx,
            'competitors_list': competitor_ids # NEW: сохраняем список id конкурентов
        }
        

        results.append(row_result)
        
    except Exception as e:
        print(f"Ошибка на индексе {idx}: {e}")
        # NEW: не забываем добавить пустой список при ошибке
        results.append({'fitness_id': idx, 'competitors_list': []})

# --- СОХРАНЕНИЕ РЕЗУЛЬТАТА ---
res_df2 = pd.DataFrame(results)
print(f"Обработка завершена за {time.time() - start:.2f} секунд.")
res_df2.head()

Загружаем пешеходный граф Москвы...
Подготовка пространственных данных...
НАЧАЛО ОБРАБОТКИ
Начинаем обработку 5029 объектов...
Обрабатываем объект 1/5029...
Ошибка на индексе 19: name 'ALL_EMPTY_METRICS' is not defined
Ошибка на индексе 38: name 'ALL_EMPTY_METRICS' is not defined
Ошибка на индексе 40: name 'ALL_EMPTY_METRICS' is not defined
Ошибка на индексе 247: name 'ALL_EMPTY_METRICS' is not defined
Обрабатываем объект 251/5029...
Ошибка на индексе 445: name 'ALL_EMPTY_METRICS' is not defined
Обрабатываем объект 501/5029...
Ошибка на индексе 724: name 'ALL_EMPTY_METRICS' is not defined
Обрабатываем объект 751/5029...
Обрабатываем объект 1001/5029...
Ошибка на индексе 1142: name 'ALL_EMPTY_METRICS' is not defined
Обрабатываем объект 1251/5029...
Обрабатываем объект 1501/5029...
Ошибка на индексе 1700: name 'ALL_EMPTY_METRICS' is not defined
Обрабатываем объект 1751/5029...
Ошибка на индексе 1996: name 'ALL_EMPTY_METRICS' is not defined
Ошибка на индексе 1998: name 'ALL_EMPTY_METRICS'

,fitness_id,competitors_list
0,0,"[12, 28, 44, 45, 164, 253, 255, 288, 301, 409,..."
1,1,"[20, 29, 32, 37, 53, 70, 123, 194, 226, 233, 2..."
2,2,"[7, 55, 65, 66, 67, 71, 102, 163, 170, 185, 19..."
3,3,"[56, 57, 95, 175, 177, 179, 221, 222, 265, 320..."
4,4,"[11, 47, 78, 79, 86, 104, 114, 115, 125, 146, ..."


### Обрабатываем конкурентов и объекты своей же сети по временному промежутку

In [228]:
res_df2.columns=['fitness_id', 'competitors_list', 'competitors_count',
       'competitors_during_all_comp',
       'competitors_during_part_comp',
       'competitors_during_all_comp_count',
       'competitors_during_part_comp_count',
       'competitors_during_all_cann',
       'competitors_during_part_cann',
       'competitors_during_all_cann_count',
       'competitors_during_part_cann_count',
       'competitors_comp_cnt', 'competitors_cann_cnt']

In [230]:
# Убираем найденные вручную дубли
numbers_to_remove = [1479, 508, 509, 510, 2156, 2386, 3827, 4883,  2702,
               84, 223,1118, 2041,2948, 3819,93,143, 209, 478, 552,
               774, 861, 1074, 1170,1292, 1581, 1879, 1893,
                 1967, 1974,2385,2946,4492]

numbers_to_remove = set([num for pair in pairs for num in pair] + ids_to_drop)

columns_to_clean = [
    'competitors_during_all_cann',
    'competitors_during_part_cann',
    'competitors_list',
    'competitors_during_all_comp',
    'competitors_during_part_comp'
]

# 3. Функция фильтрации
def filter_ids(val):
    # Проверяем, что значение действительно является списком (защита от NaN/float)
    if isinstance(val, list):
        return [x for x in val if x not in numbers_to_remove]
    return val

# 4. Применяем функцию к нужным колонкам
for col in columns_to_clean:
    res_df2[col] = res_df2[col].apply(filter_ids)

In [231]:
res_df2[['competitors_during_all_comp', 'competitors_during_part_comp', 
        'competitors_during_all_comp_count', 'competitors_during_part_comp_count',
       'competitors_during_all_cann', 'competitors_during_part_cann', 
        'competitors_during_all_cann_count', 'competitors_during_part_cann_count',]] = res_df2.apply(to_def_true_comp, axis=1)
res_df2['competitors_comp_cnt'] = res_df2['competitors_during_all_comp_count']/res_df2['competitors_count']
res_df2['competitors_cann_cnt'] = res_df2['competitors_during_all_cann_count']/res_df2['competitors_count']
# Выводим результат
res_df2.sample(10)

,fitness_id,competitors_list,competitors_count,competitors_during_all_comp,competitors_during_part_comp,competitors_during_all_comp_count,competitors_during_part_comp_count,competitors_during_all_cann,competitors_during_part_cann,competitors_during_all_cann_count,competitors_during_part_cann_count,competitors_comp_cnt,competitors_cann_cnt
32,32,"[1, 20, 29, 37, 53, 70, 123, 131, 172, 194, 22...",291,"[1, 29, 37, 70, 123, 131, 172, 194, 233, 251, ...",[],202,0,[],[],0,0,0.694158,0.000
4368,4368,"[102, 122, 185, 786, 803, 902, 907, 908, 1377,...",29,"[102, 786, 907, 908, 1377, 1852, 1901, 2114, 2...","[122, 185, 803, 902, 1388, 1452, 1475, 1881]",21,8,[],[],0,0,0.724138,0.000
190,190,"[23, 51, 89, 116, 132, 182, 183, 189, 234, 243...",99,"[89, 116, 182, 243, 632, 636, 708, 910, 1000, ...","[51, 132, 183, 189, 234, 244, 357, 862, 863, 8...",33,46,[],[],0,0,0.333333,0.000
3021,3021,"[377, 378, 411, 465, 468, 524, 594, 752, 753, ...",121,"[377, 378, 752, 753, 918, 983, 993, 1098, 1102...","[411, 465, 468, 524, 594, 1001, 1251, 1317, 13...",80,37,[],[],0,0,0.661157,0.000
1230,1230,"[49, 251, 312, 322, 323, 325, 438, 512, 557, 5...",84,"[312, 438, 512, 577, 685, 717, 1069, 1120, 114...","[251, 322, 323, 325, 888, 2424, 2966, 3300, 38...",46,10,[],[],0,0,0.547619,0.000
3830,3830,"[122, 188, 230, 231, 259, 282, 351, 410, 412, ...",108,"[188, 282, 417, 671, 687, 1377, 1569, 1713, 17...","[122, 230, 231, 351, 410, 412, 582, 786, 812, ...",25,54,[],[],0,0,0.231481,0.000
1274,1274,"[1, 16, 19, 32, 37, 38, 40, 70, 96, 101, 111, ...",185,"[16, 37, 38, 40, 111, 137, 278, 297, 298, 308,...","[1, 32, 70, 96, 101, 130, 171, 247, 248, 260, ...",64,113,[],[],0,0,0.345946,0.000
2717,2717,"[51, 89, 116, 182, 183, 189, 190, 191, 205, 23...",149,"[89, 116, 182, 243, 542, 703, 734, 910, 1000, ...","[51, 183, 189, 191, 205, 234, 357, 540, 727, 7...",38,64,[],[],0,0,0.255034,0.000
1609,1609,"[11, 12, 28, 44, 45, 85, 150, 155, 333, 484, 5...",64,"[11, 12, 28, 44, 535, 857, 1019, 1198, 1353, 1...","[45, 85, 150, 155, 333, 484, 791, 1209, 1356, ...",27,17,[],[],0,0,0.421875,0.000
4806,4806,"[204, 347, 353, 374, 416, 420, 566, 570, 571, ...",125,"[347, 353, 416, 570, 666, 1054, 1056, 1112, 11...","[374, 571, 880, 1127, 1315, 2161, 2297, 2362, ...",59,21,[4821],[],1,0,0.472000,0.008


In [232]:
res_df2.columns = ['fitness_id', 'competitors_list_15drive', 'competitors_count_15drive',
       'competitors_during_all_comp_15drive', 'competitors_during_part_comp_15drive',
       'competitors_during_all_comp_count_15drive',
       'competitors_during_part_comp_count_15drive', 'competitors_during_all_cann_15drive',
       'competitors_during_part_cann_15drive', 'competitors_during_all_cann_count_15drive',
       'competitors_during_part_cann_count_15drive', 'competitors_comp_cnt_15drive',
       'competitors_cann_cnt_15drive']

res_df2.to_excel('fitness_competitors_15drive.xlsx', index=False)
res_df2.to_csv('fitness_competitors_15drive.csv', index=False)

In [233]:
res_df2['competitors_during_all_cann_count_15drive'].value_counts()

competitors_during_all_cann_count_15drive
0     4428
1      409
2      105
3       35
4       25
5       11
7        6
6        4
8        3
10       2
9        1
Name: count, dtype: int64

In [234]:
res_df_itog = pd.concat([res_df, res_df2.iloc[:,1:]], axis =1)

res_df_itog.to_excel('fitness_competitors_15ped_drive.xlsx', index=False)
res_df_itog.to_csv('fitness_competitors_15ped_drive.csv', index=False)

In [235]:
res_df_itog.shape

(5029, 25)